## Load data

In [ ]:
import sys
import os

# Find the project root (Speciale_Kode)
current_dir = os.getcwd()
project_root = current_dir

# Looks for "Speciale_Kode" folder:
while os.path.basename(project_root) != "Speciale_Kode":
    project_root = os.path.dirname(project_root)

# Add to Python path
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
import pandas as pd
from pathlib import Path
from Modules.read_data import read_data

PRICE_ZONE = "DK1"  # "DK1" or "DK2"
TRAIN_WINDOW = 2 * 8760
VAL_START = "2024-01-01 00:00:00"
VAL_WINDOW = 8784
PREDICT_PERIOD = 4 * 168
STRIDE = 13 * 168                # Stride is measured from the start of the previous fold.
POST_VALIDATION_EXCLUDE_HOURS = 168  # Exclude first 168h after each validation window from remainder_2024_for_train
INCLUDE_REMAINING_2024_DURING_TRAINING = True
INCLUDE_DKPRICE_LAG1_AS_INPUT = True   # Add DKPrice_lag1 as an input feature
INCLUDE_PRICE_HISTORY_AS_INPUT = False   # Add actual DKPrice as an input feature (sets use_target_history=True)
INCLUDE_LAGS = False             # Include lag features (Price_lag1, Price_lag24, etc.) in training input
USE_FORECASTED_HISTORY = True    # Use model-predicted prices to compute lag features during predictionn
INCLUDE_PRICE_LAG1_AS_INPUT = True

(
    DK1_train,
    DK1_test,
    DK2_train,
    DK2_test,
    DK1_train_weather,
    DK1_test_weather,
    DK2_train_weather,
    DK2_test_weather
) = read_data("combined_data_cleaned_v5.csv")

if PRICE_ZONE == "DK1":
    dataset_train = DK1_train.copy()
    dataset_test = DK1_test.copy()
elif PRICE_ZONE == "DK2":
    dataset_train = DK2_train.copy()
    dataset_test = DK2_test.copy()
else:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")

# read_data already returns all of 2024 in dataset_train and all of 2025 in dataset_test.
# Build the custom 2024 rolling validation split entirely from dataset_train.
dataset_train = dataset_train.sort_values("Time").reset_index(drop=True)
dataset_test = dataset_test.sort_values("Time").reset_index(drop=True)

val_start_ts = pd.Timestamp(VAL_START)
year_2024_start = pd.Timestamp("2024-01-01 00:00:00")
year_2025_start = pd.Timestamp("2025-01-01 00:00:00")

# Keep legacy full timeline variable before redefining dataset_train below.
df = pd.concat([dataset_train, dataset_test], ignore_index=True).sort_values("Time").reset_index(drop=True)

# Fixed history block: TRAIN_WINDOW ending at VAL_START.
history = dataset_train.loc[dataset_train["Time"] < val_start_ts].copy().iloc[-TRAIN_WINDOW:]
if len(history) < TRAIN_WINDOW:
    raise ValueError(
        f"Not enough history for TRAIN_WINDOW={TRAIN_WINDOW}. Got {len(history)} rows before {VAL_START}."
    )

data_2024 = dataset_train.loc[
    (dataset_train["Time"] >= year_2024_start) & (dataset_train["Time"] < year_2025_start)
].copy()

validation_idx = []
validation_windows = []
window_start = val_start_ts

# Validation windows in 2024: next fold starts STRIDE hours after the current fold start.
# Only full windows are allowed; trailing partial windows are skipped.
while (window_start + pd.Timedelta(hours=PREDICT_PERIOD)) <= year_2025_start:
    window_end = window_start + pd.Timedelta(hours=PREDICT_PERIOD)
    if window_end <= window_start:
        break

    mask = (data_2024["Time"] >= window_start) & (data_2024["Time"] < window_end)
    if mask.any():
        validation_idx.extend(data_2024.index[mask].tolist())
        validation_windows.append((window_start, window_end))

    window_start = window_start + pd.Timedelta(hours=STRIDE)

validation_idx = sorted(set(validation_idx))

# Exclude first POST_VALIDATION_EXCLUDE_HOURS after each validation window from train remainder.
post_validation_exclusion_idx = []
for _, window_end in validation_windows:
    exclusion_end = min(window_end + pd.Timedelta(hours=POST_VALIDATION_EXCLUDE_HOURS), year_2025_start)
    if exclusion_end <= window_end:
        continue

    exclusion_mask = (data_2024["Time"] >= window_end) & (data_2024["Time"] < exclusion_end)
    if exclusion_mask.any():
        post_validation_exclusion_idx.extend(data_2024.index[exclusion_mask].tolist())

post_validation_exclusion_idx = sorted(set(post_validation_exclusion_idx))
excluded_from_remainder_idx = sorted(set(validation_idx).union(post_validation_exclusion_idx))

dataset_validation = data_2024.loc[validation_idx].copy().sort_values("Time").reset_index(drop=True)
remainder_2024_for_train = data_2024.drop(index=excluded_from_remainder_idx).copy().sort_values("Time").reset_index(drop=True)

# Load cell is the only place that decides whether 2024 remainder is included in training.
if INCLUDE_REMAINING_2024_DURING_TRAINING:
    dataset_train = (
        pd.concat([history, remainder_2024_for_train], ignore_index=True)
        .sort_values("Time")
        .drop_duplicates(subset=["Time"], keep="last")
        .reset_index(drop=True)
    )
else:
    dataset_train = history.copy().sort_values("Time").reset_index(drop=True)

# Full context dataset: pre-2024 history + all of 2024.
# Used by get_predictions for lag computation regardless of training flags.
dataset_context = (
    pd.concat([history, data_2024], ignore_index=True)
    .sort_values("Time")
    .drop_duplicates(subset=["Time"], keep="last")
    .reset_index(drop=True)
)

# Preserve full target-bearing datasets for training/evaluation and create input views for later cells.
dataset_train_full = dataset_train.copy()
dataset_validation_full = dataset_validation.copy()
dataset_train_input = dataset_train_full.copy()
dataset_validation_input = dataset_validation_full.copy()

if not INCLUDE_PRICE_HISTORY_AS_INPUT:
    dataset_train_input = dataset_train_input.drop(columns=["DKPrice"])
    dataset_validation_input = dataset_validation_input.drop(columns=["DKPrice"])

lag_columns = [c for c in dataset_train_input.columns if '_lag' in c]
if not INCLUDE_LAGS:
    dataset_train_input = dataset_train_input.drop(columns=lag_columns, errors='ignore')
    dataset_validation_input = dataset_validation_input.drop(
        columns=[c for c in dataset_validation_input.columns if '_lag' in c], errors='ignore'
    )

# Add DKPrice_lag1 as a feature when INCLUDE_DKPRICE_LAG1_AS_INPUT is enabled.
# Computed from the full timeline (df) to handle training-set gaps correctly.
if INCLUDE_DKPRICE_LAG1_AS_INPUT:
    price_lag_full = df[["Time", "DKPrice"]].copy().sort_values("Time").reset_index(drop=True)
    price_lag_full["DKPrice_lag1"] = price_lag_full["DKPrice"].shift(1)
    price_lag_full = price_lag_full[["Time", "DKPrice_lag1"]]
    dataset_train_input = dataset_train_input.merge(price_lag_full, on="Time", how="left")
    dataset_validation_input = dataset_validation_input.merge(price_lag_full, on="Time", how="left")

# Keep 2025 as test set.
dataset_test = dataset_test.copy().reset_index(drop=True)

target_time = val_start_ts
prices = history["DKPrice"].astype(float).values.reshape(-1, 1)

def _load_feature_predictions_for_zone(zone):
    prediction_path = Path(project_root) / "Data" / f"feature_predictions_{zone}_2024-2025.csv"
    if not prediction_path.exists():
        print("No precomputed forecasts found.")
        return None

    predictions = pd.read_csv(prediction_path, sep=";", decimal=".", parse_dates=["Time"], dayfirst=True)
    predictions = predictions.loc[:, ~predictions.columns.duplicated()].copy()
    if "DKZone" in predictions.columns:
        predictions = predictions.loc[predictions["DKZone"] == zone].copy()
        print(f"Loaded {len(predictions)} forecasts for zone {zone}.")
        print(f"Forecast features: {len(predictions.columns)} {predictions.columns.tolist()}")
    return predictions

print(f"Using zone: {PRICE_ZONE}")
print(f"Train source shape (all of 2024): {DK1_train.shape if PRICE_ZONE == 'DK1' else DK2_train.shape}")
print(f"Test source shape (all of 2025): {DK1_test.shape if PRICE_ZONE == 'DK1' else DK2_test.shape}")
print(f"Include remainder_2024_for_train in training: {INCLUDE_REMAINING_2024_DURING_TRAINING}")
print(f"Include DKPrice_lag1 as input feature: {INCLUDE_DKPRICE_LAG1_AS_INPUT}")
print(f"Include DKPrice as input feature (use_target_history): {INCLUDE_PRICE_HISTORY_AS_INPUT}")
print(f"Include lag features in training input: {INCLUDE_LAGS}")
print(f"Use forecasted prices for lag computation during prediction: {USE_FORECASTED_HISTORY}")
print(f"Train shape (prepared training dataset): {dataset_train.shape}")
print(f"Training input shape: {dataset_train_input.shape}")
print(f"2024 remainder rows included in training: {len(remainder_2024_for_train) if INCLUDE_REMAINING_2024_DURING_TRAINING else 0}")
print(f"Validation shape (rolling 2024 windows): {dataset_validation.shape}")
print(f"Validation input shape: {dataset_validation_input.shape}")
print(f"Context shape (history + all 2024): {dataset_context.shape}")
print(f"Test shape (2025): {dataset_test.shape}")
print(f"Validation windows created: {len(validation_windows)}")
print(f"Post-validation exclusion hours: {POST_VALIDATION_EXCLUDE_HOURS}")
print(f"Rows excluded from remainder after validation windows: {len(post_validation_exclusion_idx)}")
if validation_windows:
    print("All validation windows:")
    for idx, (window_start, window_end) in enumerate(validation_windows, start=1):
        print(f"  {idx:02d}. {window_start} -> {window_end}")
print(f"Training dataset columns: {dataset_train.columns.tolist()}")
print(f"Training input columns: {dataset_train_input.columns.tolist()}")

feature_predictions = None
if feature_predictions is not None:
    print("\nPrecomputed forecasts loaded.")

use_precomputed_feature_values = feature_predictions is not None

In [ ]:
from Modules.Load_RF_forecast_models import load_rf_models

rf_models = None
if not use_precomputed_feature_values:
    # load_rf_models currently supports only the optional timeout argument.
    rf_models = load_rf_models(user="Christine - laptop")      # set user to "Nikolaj" or "Christine"

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA DIAGNOSTICS")
print("\nBasic Info:")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"\nGPU Info:")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"Device Count: {torch.cuda.device_count()}")

    test_tensor = torch.randn(100, 100).to(device)
    print(f"Tensor on CUDA: {test_tensor.is_cuda}")

else:
    print("\n  Running on CPU - no CUDA available")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.base import BaseEstimator, RegressorMixin
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def smape_mean(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.abs(y_true) + np.abs(y_pred)
    vals = np.where(denom == 0, 0.0, 200.0 * np.abs(y_pred - y_true) / denom)
    return float(np.mean(vals))

class TabularSimpleRNN(nn.Module):
    """RNN over true temporal windows: (batch, sequence_length, n_features)."""

    def __init__(self, input_size: int, hidden_size: int, layers: int, dropout: float = 0.0):
        super().__init__()
        self.dropout = float(dropout)
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=layers,
            dropout=self.dropout if layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

class TorchRNNRegressor(BaseEstimator, RegressorMixin):
    """Scikit-learn style regressor using configurable rolling time sequences."""

    def __init__(
        self,
        hidden_size: int = 32,
        layers: int = 1,
        learning_rate: float = 1e-3,
        epochs: int = 40,
        batch_size: int = 64,
        sequence_length: int = 24,
        dropout: float = 0.0,
        random_state: int = 42,
        log_epoch_metrics: bool = False,
        log_prefix: str = "",
        warm_start: bool = False,
        use_target_history: bool = True,
    ):
        self.hidden_size = hidden_size
        self.layers = layers
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.sequence_length = sequence_length
        self.dropout = dropout
        self.random_state = random_state
        self.log_epoch_metrics = log_epoch_metrics
        self.log_prefix = log_prefix
        self.warm_start = warm_start
        self.use_target_history = use_target_history

    def _to_tensor_sequence(self, X, y: np.ndarray | None = None, for_inference: bool = False):
        X_np = np.asarray(X, dtype=np.float32)
        if X_np.ndim != 2:
            raise ValueError(f"Expected X with shape (n_samples, n_features), got {X_np.shape}.")

        n_samples, n_features = X_np.shape
        seq_len = max(1, int(self.sequence_length))

        if n_samples == 0:
            raise ValueError("X is empty; cannot build sequences.")

        X_seq = np.empty((n_samples, seq_len, n_features + (2 if self.use_target_history else 0)), dtype=np.float32)

        # Left-pad with the first row so each timestamp gets a full sequence window.
        pad = np.repeat(X_np[:1], repeats=seq_len - 1, axis=0)
        padded = np.vstack([pad, X_np])

        if self.use_target_history:
            if (not for_inference) and y is None:
                raise ValueError("y must be provided during training when use_target_history=True.")

            y_hist = None if y is None else np.asarray(y, dtype=np.float32).reshape(-1)
            pred_hist = np.zeros(n_samples, dtype=np.float32)

            for i in range(n_samples):
                exog_window = padded[i : i + seq_len]
                target_window = np.zeros((seq_len, 1), dtype=np.float32)
                target_mask = np.zeros((seq_len, 1), dtype=np.float32)

                for j in range(seq_len):
                    idx = i - seq_len + 1 + j
                    if idx < 0:
                        continue
                    if idx < i:
                        if y_hist is not None:
                            target_window[j, 0] = y_hist[idx]
                        else:
                            target_window[j, 0] = pred_hist[idx]
                        target_mask[j, 0] = 1.0

                X_seq[i] = np.concatenate([exog_window, target_window, target_mask], axis=1)

                if for_inference:
                    with torch.no_grad():
                        x_i = torch.tensor(X_seq[i : i + 1], dtype=torch.float32).to(self.device_)
                        pred_hist[i] = float(self.model_(x_i).item())
        else:
            for i in range(n_samples):
                X_seq[i] = padded[i : i + seq_len]

        return torch.tensor(X_seq, dtype=torch.float32)

    def _initialize_model_state(self, input_size: int):
        self.device_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.pin_memory_ = self.device_.type == "cuda"
        self.input_size_ = int(input_size)
        self.model_ = TabularSimpleRNN(
            input_size=self.input_size_,
            hidden_size=int(self.hidden_size),
            layers=int(self.layers),
            dropout=float(self.dropout),
        ).to(self.device_)
        self.loss_fn_ = nn.MSELoss()
        self.optimizer_ = torch.optim.Adam(self.model_.parameters(), lr=float(self.learning_rate))
        self.epoch_losses_ = []
        self.epoch_smapes_ = []
        self.epoch_maes_ = []
        self.epoch_rmses_ = []
        self._epochs_trained_ = 0

    def fit(self, X, y):
        set_seed(self.random_state)

        y_np = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        X_tensor = self._to_tensor_sequence(X, y=y_np, for_inference=False)
        if len(y_np) != len(X_tensor):
            raise ValueError("X and y must have the same number of rows.")
        y_tensor = torch.tensor(y_np, dtype=torch.float32)

        needs_reinit = (
            (not bool(self.warm_start))
            or (not hasattr(self, "model_"))
            or (not hasattr(self, "input_size_"))
            or (int(self.input_size_) != int(X_tensor.shape[-1]))
        )
        if needs_reinit:
            self._initialize_model_state(input_size=int(X_tensor.shape[-1]))
        elif not hasattr(self, "device_"):
            self.device_ = next(self.model_.parameters()).device
            self.pin_memory_ = self.device_.type == "cuda"

        dataset_local = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset_local, batch_size=int(self.batch_size), shuffle=True, pin_memory=self.pin_memory_)

        self.model_.train()
        for _ in range(int(self.epochs)):
            batch_losses = []
            epoch_preds = []
            epoch_targets = []

            for X_batch, y_batch in loader:
                X_batch = X_batch.to(self.device_, non_blocking=self.pin_memory_)
                y_batch = y_batch.to(self.device_, non_blocking=self.pin_memory_)
                self.optimizer_.zero_grad()
                preds = self.model_(X_batch)
                loss = self.loss_fn_(preds, y_batch)
                loss.backward()
                self.optimizer_.step()

                batch_losses.append(float(loss.item()))
                epoch_preds.append(preds.detach().cpu().numpy().reshape(-1))
                epoch_targets.append(y_batch.detach().cpu().numpy().reshape(-1))

            epoch_loss = float(np.mean(batch_losses)) if batch_losses else float("nan")
            self.epoch_losses_.append(epoch_loss)

            if epoch_preds and epoch_targets:
                y_pred_epoch = np.concatenate(epoch_preds)
                y_true_epoch = np.concatenate(epoch_targets)
                epoch_smape = smape_mean(y_true_epoch, y_pred_epoch)
                epoch_mae = float(np.mean(np.abs(y_true_epoch - y_pred_epoch)))
                epoch_rmse = float(np.sqrt(np.mean((y_true_epoch - y_pred_epoch) ** 2)))
            else:
                epoch_smape = float("nan")
                epoch_mae = float("nan")
                epoch_rmse = float("nan")

            self.epoch_smapes_.append(float(epoch_smape))
            self.epoch_maes_.append(float(epoch_mae))
            self.epoch_rmses_.append(float(epoch_rmse))

            self._epochs_trained_ += 1

            if bool(self.log_epoch_metrics):
                try:
                    import wandb
                    if wandb.run is not None:
                        loss_name = f"{self.log_prefix}train_MSE_loss" if self.log_prefix else "train_MSE_loss"
                        smape_name = f"{self.log_prefix}train_smape" if self.log_prefix else "train_smape"
                        mae_name = f"{self.log_prefix}train_mae" if self.log_prefix else "train_mae"
                        rmse_name = f"{self.log_prefix}train_rmse" if self.log_prefix else "train_rmse"
                        epoch_name = f"{self.log_prefix}epoch" if self.log_prefix else "epoch"
                        wandb.log({
                            loss_name: epoch_loss,
                            smape_name: float(epoch_smape),
                            mae_name: float(epoch_mae),
                            rmse_name: float(epoch_rmse),
                            epoch_name: int(self._epochs_trained_),
                        })
                except Exception:
                    pass

        return self

    def predict(self, X):
        X_np = np.asarray(X, dtype=np.float32)
        if X_np.ndim == 2:
            self.model_.eval()
            X_tensor = self._to_tensor_sequence(X_np, y=None, for_inference=True)
        elif X_np.ndim == 3:
            X_tensor = torch.tensor(X_np, dtype=torch.float32)
        else:
            raise ValueError(f"Expected X with shape (n_samples, n_features) or (n_samples, sequence_length, n_features), got {X_np.shape}.")
        self.model_.eval()
        with torch.no_grad():
            X_tensor = X_tensor.to(self.device_, non_blocking=self.pin_memory_)
            preds = self.model_(X_tensor).squeeze(-1).detach().cpu().numpy()
        return preds

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import wandb
import joblib
from sklearn.preprocessing import StandardScaler
from Modules.Validation3 import _build_validation_folds
from Modules.week_predictions2_noise import get_predictions

PREDICT_PERIOD = 4 * 168
MAX_EPOCHS = 300
PATIENCE = 30
MIN_DELTA = 0.0
WANDB_PROJECT = "Simple_RNN_rerun"
WANDB_RUN_NAME = f"{PRICE_ZONE}_RNN_{TRAIN_WINDOW//8760}y_" + \
                    f"price_lag1{'incl' if INCLUDE_PRICE_HISTORY_AS_INPUT else 'excl'}_2024" + \
                    f"{'incl' if INCLUDE_REMAINING_2024_DURING_TRAINING else 'excl'}" + \
                    f"_{PREDICT_PERIOD//168}val"

# =====================================================================================
# Test final model on 2025 only, week-by-week (168h blocks)
# =====================================================================================
WANDB_PROJECT = "Simple_RNN_rerun"
WANDB_RUN_NAME = WANDB_RUN_NAME  # Must match final training run name
WANDB_TEST_RUN_NAME = f"RNN_noise_test"
WANDB_ARTIFACT_NAME = f"{WANDB_RUN_NAME}_model"

FORECAST_HORIZON = 168  # 1 week
PREDICT_PERIOD = 168    # one week per validation fold
STRIDE = 168            # next fold starts next week
TEST_START = pd.Timestamp("2025-01-01 00:00:00")
TEST_WINDOW = 8760
TEST_END = TEST_START + pd.Timedelta(hours=TEST_WINDOW - 1)

# Retrieve zone data
if PRICE_ZONE == "DK1":
    test_source = DK1_test.copy() if "DK1_test" in globals() else None
    history_source = DK1_train.copy() if "DK1_train" in globals() else None
elif PRICE_ZONE == "DK2":
    test_source = DK2_test.copy() if "DK2_test" in globals() else None
    history_source = DK2_train.copy() if "DK2_train" in globals() else None
else:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")

if test_source is None or history_source is None:
    raise ValueError(f"Missing train/test source data for {PRICE_ZONE}. Run data loading first.")

test_source = test_source.sort_values("Time").reset_index(drop=True)
history_source = history_source.sort_values("Time").reset_index(drop=True)

# Test only on 2025 window requested by user
test_set = test_source.loc[
    (test_source["Time"] >= TEST_START) & (test_source["Time"] <= TEST_END)
].copy().sort_values("Time").reset_index(drop=True)

if len(test_set) != TEST_WINDOW:
    raise ValueError(
        f"Expected exactly TEST_WINDOW={TEST_WINDOW} rows from {TEST_START} to {TEST_END}, got {len(test_set)}."
    )

# Use the same feature columns the model was trained on (respects INCLUDE_LAGS and INCLUDE_PRICE_HISTORY_AS_INPUT)
feature_columns = [c for c in dataset_train_input.columns if c not in ["Time", "DKPrice"]]
if not feature_columns:
    raise ValueError("No feature columns found in dataset_train_input. Run the load data cell first.")

# Include end-of-2024 history so first 2025 prediction can build sequence/lag features.
# get_predictions expects DKPrice as the first column (target), followed by Time and features.
full_eval_dataset = (
    pd.concat([history_source, test_set], ignore_index=True)
    .sort_values("Time")
    .drop_duplicates(subset=["Time"], keep="last")
    .reset_index(drop=True)
)
# Compute DKPrice_lag1 if the model was trained with it as a feature.
if "DKPrice_lag1" in feature_columns:
    full_eval_dataset["DKPrice_lag1"] = full_eval_dataset["DKPrice"].shift(1)
missing_in_eval = [c for c in feature_columns if c not in full_eval_dataset.columns]
if missing_in_eval:
    raise ValueError(f"full_eval_dataset is missing feature columns from dataset_train_input: {missing_in_eval}")
# DKPrice must be the first column so get_predictions treats it as the target
full_eval_dataset = full_eval_dataset[["DKPrice", "Time"] + feature_columns].copy()

# Initialize W&B run
try:
    wandb.finish()
except Exception:
    pass
test_run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_TEST_RUN_NAME,
    job_type="evaluation",
    config={
        "price_zone": PRICE_ZONE,
        "test_start": str(TEST_START),
        "test_end": str(TEST_END),
        "test_window": int(TEST_WINDOW),
        "test_rows": int(len(test_set)),
        "model_artifact": WANDB_ARTIFACT_NAME,
        "forecast_horizon": int(FORECAST_HORIZON),
        "predict_period": int(PREDICT_PERIOD),
        "stride": int(STRIDE),
        "prediction_method": "weekly_168h_autoregressive",
    },
    tags=["rnn", "final-model", "test", "evaluation", "weekly-metrics"],
    reinit=True,
    settings=wandb.Settings(start_method="thread"),
)

# Load trained model
model_artifact = test_run.use_artifact(f"{WANDB_ARTIFACT_NAME}:latest")
artifact_dir = Path(model_artifact.download())
model_path = artifact_dir / "model.joblib"
if not model_path.exists():
    raise ValueError(f"Could not find model.joblib in downloaded artifact: {artifact_dir}")

model = joblib.load(model_path)
print(f"Loaded model artifact: {WANDB_ARTIFACT_NAME}:latest")
print(f"Test window: {TEST_START} -> {TEST_END} ({len(test_set)} rows)")
print(f"Feature columns ({len(feature_columns)}): {feature_columns}")

# Fit scaler on the same training tail used during training.
# Compute target-lag columns on the full sorted history before taking the tail
# so that the first row of the tail has a valid lag value.
train_for_scaler = history_source.sort_values("Time").copy()
if "DKPrice_lag1" in feature_columns and "DKPrice_lag1" not in train_for_scaler.columns:
    train_for_scaler["DKPrice_lag1"] = train_for_scaler["DKPrice"].shift(1)
train_for_scaler = train_for_scaler.tail(TRAIN_WINDOW).reset_index(drop=True)
X_scaler = train_for_scaler[feature_columns].astype(np.float32)
scaler = StandardScaler()
scaler.fit(X_scaler)

# Build 2025 weekly folds
folds = _build_validation_folds(
    data=full_eval_dataset,
    val_window=TEST_WINDOW,
    val_start=str(TEST_START),
    predict_period=PREDICT_PERIOD,
    stride=STRIDE,
    validation_reference=test_set,
)
print(f"Generated {len(folds)} weekly folds with predict_period={PREDICT_PERIOD}, stride={STRIDE}.")

# Metric helper
def smape_mean(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.abs(y_true) + np.abs(y_pred)
    vals = np.where(denom == 0, 0.0, 200.0 * np.abs(y_pred - y_true) / denom)
    return float(np.mean(vals))

# Evaluate fold-by-fold (week-by-week)
all_week_evals = []
day_smapes_per_week = {}  # (fold_no, week_no) -> {day_num -> smape}

for fold in folds:
    fold_no = int(fold["fold"])
    fold_val_data = test_set.loc[
        (test_set["Time"] >= fold["val_start"]) & (test_set["Time"] <= fold["val_end"])
    ].copy()

    preds_by_week = get_predictions(
        model=model,
        dataset=full_eval_dataset,
        val_start=fold["val_start"],
        val_end=fold["val_end"],
        forecast_horizon=FORECAST_HORIZON,
        fitted_scaler=scaler,
        dk_zone=PRICE_ZONE,
        rf_models=rf_models if "rf_models" in globals() else None,
        use_precomputed_feature_values=use_precomputed_feature_values if "use_precomputed_feature_values" in globals() else False,
        precomputed_feature_predictions=feature_predictions if "feature_predictions" in globals() else None,
        use_forecasted_history=USE_FORECASTED_HISTORY,
    )

    for week_no, week_pred_df in preds_by_week.items():
        week_eval = week_pred_df.merge(
            fold_val_data[["Time", "DKPrice"]],
            on="Time",
            how="left",
        )
        week_eval = week_eval.dropna(subset=["DKPrice"]).copy()
        if week_eval.empty:
            continue

        week_eval["fold"] = fold_no
        week_eval["week_in_fold"] = int(week_no)
        all_week_evals.append(week_eval)

        week_key = (fold_no, int(week_no))
        day_smapes_per_week[week_key] = {}
        week_eval_sorted = week_eval.sort_values("Time").reset_index(drop=True)

        for day_num in range(1, 8):
            start_hour = (day_num - 1) * 24
            end_hour = day_num * 24
            day_data = week_eval_sorted.iloc[start_hour:end_hour]
            if len(day_data) > 0:
                day_smapes_per_week[week_key][day_num] = smape_mean(
                    day_data["DKPrice"].values,
                    day_data["Prediction"].values,
                )

if not all_week_evals:
    raise RuntimeError("No aligned predictions produced for 2025 weekly folds.")

results_df = pd.concat(all_week_evals, ignore_index=True).sort_values("Time").reset_index(drop=True)
results_df = results_df.rename(columns={"DKPrice": "Actual"})
results_df["Error"] = results_df["Actual"] - results_df["Prediction"]
results_df["Abs_Error"] = np.abs(results_df["Error"])

y_actual = results_df["Actual"].values
y_pred = results_df["Prediction"].values

# Overall metrics
overall_mse = float(np.mean((y_actual - y_pred) ** 2))
overall_smape = smape_mean(y_actual, y_pred)
overall_mae = float(np.mean(np.abs(y_actual - y_pred)))
overall_rmse = float(np.sqrt(overall_mse))

print("\n=== Overall Test Results ===")
print(f"MSE Loss:  {overall_mse:.6f}")
print(f"SMAPE:     {overall_smape:.6f}")
print(f"MAE:       {overall_mae:.6f}")
print(f"RMSE:      {overall_rmse:.6f}")

# Weekly metrics (calendar week)
print("\n=== Weekly Metrics ===")
results_df["Week"] = results_df["Time"].dt.isocalendar().week
weekly_results = []
for week, group in results_df.groupby("Week"):
    week_actual = group["Actual"].values
    week_pred = group["Prediction"].values
    week_rmse = float(np.sqrt(np.mean((week_actual - week_pred) ** 2)))
    week_mae = float(np.mean(np.abs(week_actual - week_pred)))
    week_smape = smape_mean(week_actual, week_pred)
    weekly_results.append({
        "week": int(week),
        "rmse": week_rmse,
        "mae": week_mae,
        "smape": week_smape,
        "n_samples": len(group),
    })
    print(f"Week {week:02d}: RMSE={week_rmse:.4f}, MAE={week_mae:.4f}, SMAPE={week_smape:.4f}")

# Daily metrics
print("\n=== Daily Metrics (by Date) ===")
results_df["Date"] = results_df["Time"].dt.date
daily_results = []
for date, group in results_df.groupby("Date"):
    day_actual = group["Actual"].values
    day_pred = group["Prediction"].values
    day_rmse = float(np.sqrt(np.mean((day_actual - day_pred) ** 2)))
    day_mae = float(np.mean(np.abs(day_actual - day_pred)))
    day_smape = smape_mean(day_actual, day_pred)
    daily_results.append({
        "date": str(date),
        "rmse": day_rmse,
        "mae": day_mae,
        "smape": day_smape,
        "n_samples": len(group),
    })
    print(f"{date}: RMSE={day_rmse:.4f}, MAE={day_mae:.4f}, SMAPE={day_smape:.4f}")

# Day-of-forecast metrics (Day 1..7 inside each predicted week)
print("\n=== Day-of-Forecast Metrics (within each 168-hour period) ===")
day_of_forecast_smapes = {}
for day_num in range(1, 8):
    smapes = [
        day_smapes_per_week[w].get(day_num)
        for w in day_smapes_per_week
        if day_num in day_smapes_per_week[w]
    ]
    smapes = [s for s in smapes if s is not None and not np.isnan(s)]
    day_of_forecast_smapes[day_num] = float(np.mean(smapes)) if smapes else float("nan")
    print(f"Day {day_num} (hours {(day_num-1)*24}-{day_num*24-1}): avg SMAPE={day_of_forecast_smapes[day_num]:.4f}")

# Average weekly/daily metrics
avg_weekly_rmse = float(np.mean([r["rmse"] for r in weekly_results]))
avg_weekly_mae = float(np.mean([r["mae"] for r in weekly_results]))
avg_weekly_smape = float(np.mean([r["smape"] for r in weekly_results]))

avg_daily_rmse = float(np.mean([r["rmse"] for r in daily_results]))
avg_daily_mae = float(np.mean([r["mae"] for r in daily_results]))
avg_daily_smape = float(np.mean([r["smape"] for r in daily_results]))

print("\n=== Average Weekly Metrics ===")
print(f"Avg Weekly RMSE: {avg_weekly_rmse:.6f}")
print(f"Avg Weekly MAE:  {avg_weekly_mae:.6f}")
print(f"Avg Weekly SMAPE: {avg_weekly_smape:.6f}")

print("\n=== Average Daily Metrics ===")
print(f"Avg Daily RMSE: {avg_daily_rmse:.6f}")
print(f"Avg Daily MAE:  {avg_daily_mae:.6f}")
print(f"Avg Daily SMAPE: {avg_daily_smape:.6f}")

# Save all hourly predictions to CSV
output_root = Path(project_root) / "Deep learners" / "Simple RNN"
output_root.mkdir(parents=True, exist_ok=True)
predictions_csv_path = output_root / f"{WANDB_RUN_NAME}_test_predictions.csv"
results_df.to_csv(predictions_csv_path, index=False, sep=";", decimal=".")
print(f"\nHourly predictions saved to: {predictions_csv_path}")

# Log metrics
test_run.log({
    "test_overall_MSE_loss": overall_mse,
    "test_overall_SMAPE": overall_smape,
    "test_overall_MAE": overall_mae,
    "test_overall_RMSE": overall_rmse,
    "test_avg_weekly_RMSE": avg_weekly_rmse,
    "test_avg_weekly_MAE": avg_weekly_mae,
    "test_avg_weekly_SMAPE": avg_weekly_smape,
    "test_avg_daily_RMSE": avg_daily_rmse,
    "test_avg_daily_MAE": avg_daily_mae,
    "test_avg_daily_SMAPE": avg_daily_smape,
    "test_avg_smape_day_1": day_of_forecast_smapes.get(1, float("nan")),
    "test_avg_smape_day_2": day_of_forecast_smapes.get(2, float("nan")),
    "test_avg_smape_day_3": day_of_forecast_smapes.get(3, float("nan")),
    "test_avg_smape_day_4": day_of_forecast_smapes.get(4, float("nan")),
    "test_avg_smape_day_5": day_of_forecast_smapes.get(5, float("nan")),
    "test_avg_smape_day_6": day_of_forecast_smapes.get(6, float("nan")),
    "test_avg_smape_day_7": day_of_forecast_smapes.get(7, float("nan")),
})

weekly_df = pd.DataFrame(weekly_results)
daily_df = pd.DataFrame(daily_results)
test_run.log({
    "test_weekly_metrics": wandb.Table(dataframe=weekly_df),
    "test_daily_metrics": wandb.Table(dataframe=daily_df),
    "test_predictions_sample": wandb.Table(dataframe=results_df.head(100)),
})

test_run.summary.update({
    "test_overall_MSE_loss": overall_mse,
    "test_overall_SMAPE": overall_smape,
    "test_overall_MAE": overall_mae,
    "test_overall_RMSE": overall_rmse,
    "test_avg_weekly_RMSE": avg_weekly_rmse,
    "test_avg_weekly_MAE": avg_weekly_mae,
    "test_avg_weekly_SMAPE": avg_weekly_smape,
    "test_avg_daily_RMSE": avg_daily_rmse,
    "test_avg_daily_MAE": avg_daily_mae,
    "test_avg_daily_SMAPE": avg_daily_smape,
    "test_samples": int(len(results_df)),
    "test_period_start": str(TEST_START),
    "test_period_end": str(TEST_END),
    "n_weeks": len(weekly_results),
    "n_days": len(daily_results),
})


print(f"\nTest predictions and metrics logged to W&B run: {WANDB_TEST_RUN_NAME}")
wandb.finish()


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import wandb
import joblib
from sklearn.preprocessing import StandardScaler
from Modules.Validation3 import _build_validation_folds
from Modules.week_predictions2_noise import get_predictions

PREDICT_PERIOD = 4 * 168
MAX_EPOCHS = 300
PATIENCE = 40
MIN_DELTA = 0.0
WANDB_PROJECT = "GRU_no_precompute"
WANDB_RUN_NAME = f"{PRICE_ZONE}_GRU_{TRAIN_WINDOW//8760}y_" + \
                    f"price_lag1{'incl' if INCLUDE_PRICE_HISTORY_AS_INPUT else 'excl'}_2024" + \
                    f"{'incl' if INCLUDE_REMAINING_2024_DURING_TRAINING else 'excl'}" + \
                    f"_{PREDICT_PERIOD//168}val"

# =====================================================================================
# Test final model on 2025 only, week-by-week (168h blocks)
# =====================================================================================
WANDB_PROJECT = "GRU_no_precompute"
WANDB_RUN_NAME = WANDB_RUN_NAME  # Must match final training run name
WANDB_TEST_RUN_NAME = f"GRU_noise_test"
WANDB_ARTIFACT_NAME = f"{WANDB_RUN_NAME}_model"

FORECAST_HORIZON = 168  # 1 week
PREDICT_PERIOD = 168    # one week per validation fold
STRIDE = 168            # next fold starts next week
TEST_START = pd.Timestamp("2025-01-01 00:00:00")
TEST_WINDOW = 8760
TEST_END = TEST_START + pd.Timedelta(hours=TEST_WINDOW - 1)

# Retrieve zone data
if PRICE_ZONE == "DK1":
    test_source = DK1_test.copy() if "DK1_test" in globals() else None
    history_source = DK1_train.copy() if "DK1_train" in globals() else None
elif PRICE_ZONE == "DK2":
    test_source = DK2_test.copy() if "DK2_test" in globals() else None
    history_source = DK2_train.copy() if "DK2_train" in globals() else None
else:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")

if test_source is None or history_source is None:
    raise ValueError(f"Missing train/test source data for {PRICE_ZONE}. Run data loading first.")

test_source = test_source.sort_values("Time").reset_index(drop=True)
history_source = history_source.sort_values("Time").reset_index(drop=True)

# Test only on 2025 window requested by user
test_set = test_source.loc[
    (test_source["Time"] >= TEST_START) & (test_source["Time"] <= TEST_END)
].copy().sort_values("Time").reset_index(drop=True)

if len(test_set) != TEST_WINDOW:
    raise ValueError(
        f"Expected exactly TEST_WINDOW={TEST_WINDOW} rows from {TEST_START} to {TEST_END}, got {len(test_set)}."
    )

# Use the same feature columns the model was trained on (respects INCLUDE_LAGS and INCLUDE_PRICE_HISTORY_AS_INPUT)
feature_columns = [c for c in dataset_train_input.columns if c not in ["Time", "DKPrice"]]
if not feature_columns:
    raise ValueError("No feature columns found in dataset_train_input. Run the load data cell first.")

# Include end-of-2024 history so first 2025 prediction can build sequence/lag features.
# get_predictions expects DKPrice as the first column (target), followed by Time and features.
full_eval_dataset = (
    pd.concat([history_source, test_set], ignore_index=True)
    .sort_values("Time")
    .drop_duplicates(subset=["Time"], keep="last")
    .reset_index(drop=True)
)
# Compute DKPrice_lag1 if the model was trained with it as a feature.
if "DKPrice_lag1" in feature_columns:
    full_eval_dataset["DKPrice_lag1"] = full_eval_dataset["DKPrice"].shift(1)
missing_in_eval = [c for c in feature_columns if c not in full_eval_dataset.columns]
if missing_in_eval:
    raise ValueError(f"full_eval_dataset is missing feature columns from dataset_train_input: {missing_in_eval}")
# DKPrice must be the first column so get_predictions treats it as the target
full_eval_dataset = full_eval_dataset[["DKPrice", "Time"] + feature_columns].copy()

# Initialize W&B run
try:
    wandb.finish()
except Exception:
    pass
test_run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_TEST_RUN_NAME,
    job_type="evaluation",
    config={
        "price_zone": PRICE_ZONE,
        "test_start": str(TEST_START),
        "test_end": str(TEST_END),
        "test_window": int(TEST_WINDOW),
        "test_rows": int(len(test_set)),
        "model_artifact": WANDB_ARTIFACT_NAME,
        "forecast_horizon": int(FORECAST_HORIZON),
        "predict_period": int(PREDICT_PERIOD),
        "stride": int(STRIDE),
        "prediction_method": "weekly_168h_autoregressive",
    },
    tags=["rnn", "final-model", "test", "evaluation", "weekly-metrics"],
    reinit=True,
    settings=wandb.Settings(start_method="thread"),
)

# Load trained model
model_artifact = test_run.use_artifact(f"{WANDB_ARTIFACT_NAME}:latest")
artifact_dir = Path(model_artifact.download())
model_path = artifact_dir / "model.joblib"
if not model_path.exists():
    raise ValueError(f"Could not find model.joblib in downloaded artifact: {artifact_dir}")

model = joblib.load(model_path)
print(f"Loaded model artifact: {WANDB_ARTIFACT_NAME}:latest")
print(f"Test window: {TEST_START} -> {TEST_END} ({len(test_set)} rows)")
print(f"Feature columns ({len(feature_columns)}): {feature_columns}")

# Fit scaler on the same training tail used during training.
# Compute target-lag columns on the full sorted history before taking the tail
# so that the first row of the tail has a valid lag value.
train_for_scaler = history_source.sort_values("Time").copy()
if "DKPrice_lag1" in feature_columns and "DKPrice_lag1" not in train_for_scaler.columns:
    train_for_scaler["DKPrice_lag1"] = train_for_scaler["DKPrice"].shift(1)
train_for_scaler = train_for_scaler.tail(TRAIN_WINDOW).reset_index(drop=True)
X_scaler = train_for_scaler[feature_columns].astype(np.float32)
scaler = StandardScaler()
scaler.fit(X_scaler)

# Build 2025 weekly folds
folds = _build_validation_folds(
    data=full_eval_dataset,
    val_window=TEST_WINDOW,
    val_start=str(TEST_START),
    predict_period=PREDICT_PERIOD,
    stride=STRIDE,
    validation_reference=test_set,
)
print(f"Generated {len(folds)} weekly folds with predict_period={PREDICT_PERIOD}, stride={STRIDE}.")

# Metric helper
def smape_mean(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.abs(y_true) + np.abs(y_pred)
    vals = np.where(denom == 0, 0.0, 200.0 * np.abs(y_pred - y_true) / denom)
    return float(np.mean(vals))

# Evaluate fold-by-fold (week-by-week)
all_week_evals = []
day_smapes_per_week = {}  # (fold_no, week_no) -> {day_num -> smape}

for fold in folds:
    fold_no = int(fold["fold"])
    fold_val_data = test_set.loc[
        (test_set["Time"] >= fold["val_start"]) & (test_set["Time"] <= fold["val_end"])
    ].copy()

    preds_by_week = get_predictions(
        model=model,
        dataset=full_eval_dataset,
        val_start=fold["val_start"],
        val_end=fold["val_end"],
        forecast_horizon=FORECAST_HORIZON,
        fitted_scaler=scaler,
        dk_zone=PRICE_ZONE,
        rf_models=rf_models if "rf_models" in globals() else None,
        use_precomputed_feature_values=use_precomputed_feature_values if "use_precomputed_feature_values" in globals() else False,
        precomputed_feature_predictions=feature_predictions if "feature_predictions" in globals() else None,
        use_forecasted_history=USE_FORECASTED_HISTORY,
    )

    for week_no, week_pred_df in preds_by_week.items():
        week_eval = week_pred_df.merge(
            fold_val_data[["Time", "DKPrice"]],
            on="Time",
            how="left",
        )
        week_eval = week_eval.dropna(subset=["DKPrice"]).copy()
        if week_eval.empty:
            continue

        week_eval["fold"] = fold_no
        week_eval["week_in_fold"] = int(week_no)
        all_week_evals.append(week_eval)

        week_key = (fold_no, int(week_no))
        day_smapes_per_week[week_key] = {}
        week_eval_sorted = week_eval.sort_values("Time").reset_index(drop=True)

        for day_num in range(1, 8):
            start_hour = (day_num - 1) * 24
            end_hour = day_num * 24
            day_data = week_eval_sorted.iloc[start_hour:end_hour]
            if len(day_data) > 0:
                day_smapes_per_week[week_key][day_num] = smape_mean(
                    day_data["DKPrice"].values,
                    day_data["Prediction"].values,
                )

if not all_week_evals:
    raise RuntimeError("No aligned predictions produced for 2025 weekly folds.")

results_df = pd.concat(all_week_evals, ignore_index=True).sort_values("Time").reset_index(drop=True)
results_df = results_df.rename(columns={"DKPrice": "Actual"})
results_df["Error"] = results_df["Actual"] - results_df["Prediction"]
results_df["Abs_Error"] = np.abs(results_df["Error"])

y_actual = results_df["Actual"].values
y_pred = results_df["Prediction"].values

# Overall metrics
overall_mse = float(np.mean((y_actual - y_pred) ** 2))
overall_smape = smape_mean(y_actual, y_pred)
overall_mae = float(np.mean(np.abs(y_actual - y_pred)))
overall_rmse = float(np.sqrt(overall_mse))

print("\n=== Overall Test Results ===")
print(f"MSE Loss:  {overall_mse:.6f}")
print(f"SMAPE:     {overall_smape:.6f}")
print(f"MAE:       {overall_mae:.6f}")
print(f"RMSE:      {overall_rmse:.6f}")

# Weekly metrics (calendar week)
print("\n=== Weekly Metrics ===")
results_df["Week"] = results_df["Time"].dt.isocalendar().week
weekly_results = []
for week, group in results_df.groupby("Week"):
    week_actual = group["Actual"].values
    week_pred = group["Prediction"].values
    week_rmse = float(np.sqrt(np.mean((week_actual - week_pred) ** 2)))
    week_mae = float(np.mean(np.abs(week_actual - week_pred)))
    week_smape = smape_mean(week_actual, week_pred)
    weekly_results.append({
        "week": int(week),
        "rmse": week_rmse,
        "mae": week_mae,
        "smape": week_smape,
        "n_samples": len(group),
    })
    print(f"Week {week:02d}: RMSE={week_rmse:.4f}, MAE={week_mae:.4f}, SMAPE={week_smape:.4f}")

# Daily metrics
print("\n=== Daily Metrics (by Date) ===")
results_df["Date"] = results_df["Time"].dt.date
daily_results = []
for date, group in results_df.groupby("Date"):
    day_actual = group["Actual"].values
    day_pred = group["Prediction"].values
    day_rmse = float(np.sqrt(np.mean((day_actual - day_pred) ** 2)))
    day_mae = float(np.mean(np.abs(day_actual - day_pred)))
    day_smape = smape_mean(day_actual, day_pred)
    daily_results.append({
        "date": str(date),
        "rmse": day_rmse,
        "mae": day_mae,
        "smape": day_smape,
        "n_samples": len(group),
    })
    print(f"{date}: RMSE={day_rmse:.4f}, MAE={day_mae:.4f}, SMAPE={day_smape:.4f}")

# Day-of-forecast metrics (Day 1..7 inside each predicted week)
print("\n=== Day-of-Forecast Metrics (within each 168-hour period) ===")
day_of_forecast_smapes = {}
for day_num in range(1, 8):
    smapes = [
        day_smapes_per_week[w].get(day_num)
        for w in day_smapes_per_week
        if day_num in day_smapes_per_week[w]
    ]
    smapes = [s for s in smapes if s is not None and not np.isnan(s)]
    day_of_forecast_smapes[day_num] = float(np.mean(smapes)) if smapes else float("nan")
    print(f"Day {day_num} (hours {(day_num-1)*24}-{day_num*24-1}): avg SMAPE={day_of_forecast_smapes[day_num]:.4f}")

# Average weekly/daily metrics
avg_weekly_rmse = float(np.mean([r["rmse"] for r in weekly_results]))
avg_weekly_mae = float(np.mean([r["mae"] for r in weekly_results]))
avg_weekly_smape = float(np.mean([r["smape"] for r in weekly_results]))

avg_daily_rmse = float(np.mean([r["rmse"] for r in daily_results]))
avg_daily_mae = float(np.mean([r["mae"] for r in daily_results]))
avg_daily_smape = float(np.mean([r["smape"] for r in daily_results]))

print("\n=== Average Weekly Metrics ===")
print(f"Avg Weekly RMSE: {avg_weekly_rmse:.6f}")
print(f"Avg Weekly MAE:  {avg_weekly_mae:.6f}")
print(f"Avg Weekly SMAPE: {avg_weekly_smape:.6f}")

print("\n=== Average Daily Metrics ===")
print(f"Avg Daily RMSE: {avg_daily_rmse:.6f}")
print(f"Avg Daily MAE:  {avg_daily_mae:.6f}")
print(f"Avg Daily SMAPE: {avg_daily_smape:.6f}")

# Save all hourly predictions to CSV
output_root = Path(project_root) / "Deep learners" / "GRU"
output_root.mkdir(parents=True, exist_ok=True)
predictions_csv_path = output_root / f"{WANDB_RUN_NAME}_test_predictions.csv"
results_df.to_csv(predictions_csv_path, index=False, sep=";", decimal=".")
print(f"\nHourly predictions saved to: {predictions_csv_path}")

# Log metrics
test_run.log({
    "test_overall_MSE_loss": overall_mse,
    "test_overall_SMAPE": overall_smape,
    "test_overall_MAE": overall_mae,
    "test_overall_RMSE": overall_rmse,
    "test_avg_weekly_RMSE": avg_weekly_rmse,
    "test_avg_weekly_MAE": avg_weekly_mae,
    "test_avg_weekly_SMAPE": avg_weekly_smape,
    "test_avg_daily_RMSE": avg_daily_rmse,
    "test_avg_daily_MAE": avg_daily_mae,
    "test_avg_daily_SMAPE": avg_daily_smape,
    "test_avg_smape_day_1": day_of_forecast_smapes.get(1, float("nan")),
    "test_avg_smape_day_2": day_of_forecast_smapes.get(2, float("nan")),
    "test_avg_smape_day_3": day_of_forecast_smapes.get(3, float("nan")),
    "test_avg_smape_day_4": day_of_forecast_smapes.get(4, float("nan")),
    "test_avg_smape_day_5": day_of_forecast_smapes.get(5, float("nan")),
    "test_avg_smape_day_6": day_of_forecast_smapes.get(6, float("nan")),
    "test_avg_smape_day_7": day_of_forecast_smapes.get(7, float("nan")),
})

weekly_df = pd.DataFrame(weekly_results)
daily_df = pd.DataFrame(daily_results)
test_run.log({
    "test_weekly_metrics": wandb.Table(dataframe=weekly_df),
    "test_daily_metrics": wandb.Table(dataframe=daily_df),
    "test_predictions_sample": wandb.Table(dataframe=results_df.head(100)),
})

test_run.summary.update({
    "test_overall_MSE_loss": overall_mse,
    "test_overall_SMAPE": overall_smape,
    "test_overall_MAE": overall_mae,
    "test_overall_RMSE": overall_rmse,
    "test_avg_weekly_RMSE": avg_weekly_rmse,
    "test_avg_weekly_MAE": avg_weekly_mae,
    "test_avg_weekly_SMAPE": avg_weekly_smape,
    "test_avg_daily_RMSE": avg_daily_rmse,
    "test_avg_daily_MAE": avg_daily_mae,
    "test_avg_daily_SMAPE": avg_daily_smape,
    "test_samples": int(len(results_df)),
    "test_period_start": str(TEST_START),
    "test_period_end": str(TEST_END),
    "n_weeks": len(weekly_results),
    "n_days": len(daily_results),
})


print(f"\nTest predictions and metrics logged to W&B run: {WANDB_TEST_RUN_NAME}")
wandb.finish()


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import wandb
import joblib
from sklearn.preprocessing import StandardScaler
from Modules.Validation3 import _build_validation_folds
from Modules.week_predictions2_noise import get_predictions

# =====================================================================================
# Test final model on 2025 only, week-by-week (168h blocks)
# =====================================================================================
WANDB_PROJECT = "LSTM final"
WANDB_RUN_NAME = "DK1_LSTM_2y_MaskExcl_Lag1Incl_2024Incl_4val_hidden64_layers3_bs64_seq24_drop0.2"  # Must match final training run name
WANDB_TEST_RUN_NAME = f"LSTM_noise_test"
WANDB_ARTIFACT_NAME = f"{WANDB_RUN_NAME}_model"


FORECAST_HORIZON = 168  # 1 week
PREDICT_PERIOD = 168    # one week per validation fold
STRIDE = 168            # next fold starts next week
TEST_START = pd.Timestamp("2025-01-01 00:00:00")
TEST_WINDOW = 8760
TEST_END = TEST_START + pd.Timedelta(hours=TEST_WINDOW - 1)

# Retrieve zone data
if PRICE_ZONE == "DK1":
    test_source = DK1_test.copy() if "DK1_test" in globals() else None
    history_source = DK1_train.copy() if "DK1_train" in globals() else None
elif PRICE_ZONE == "DK2":
    test_source = DK2_test.copy() if "DK2_test" in globals() else None
    history_source = DK2_train.copy() if "DK2_train" in globals() else None
else:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")

if test_source is None or history_source is None:
    raise ValueError(f"Missing train/test source data for {PRICE_ZONE}. Run data loading first.")

test_source = test_source.sort_values("Time").reset_index(drop=True)
history_source = history_source.sort_values("Time").reset_index(drop=True)

# Test only on 2025 window requested by user
test_set = test_source.loc[
    (test_source["Time"] >= TEST_START) & (test_source["Time"] <= TEST_END)
].copy().sort_values("Time").reset_index(drop=True)

if len(test_set) != TEST_WINDOW:
    raise ValueError(
        f"Expected exactly TEST_WINDOW={TEST_WINDOW} rows from {TEST_START} to {TEST_END}, got {len(test_set)}."
    )

# Use the same feature columns the model was trained on (respects INCLUDE_LAGS and INCLUDE_PRICE_HISTORY_AS_INPUT)
feature_columns = [c for c in dataset_train_input.columns if c not in ["Time", "DKPrice"]]
if not feature_columns:
    raise ValueError("No feature columns found in dataset_train_input. Run the load data cell first.")

# Include end-of-2024 history so first 2025 prediction can build sequence/lag features.
# get_predictions expects DKPrice as the first column (target), followed by Time and features.
full_eval_dataset = (
    pd.concat([history_source, test_set], ignore_index=True)
    .sort_values("Time")
    .drop_duplicates(subset=["Time"], keep="last")
    .reset_index(drop=True)
)
# Compute DKPrice_lag1 if the model was trained with it as a feature.
if "DKPrice_lag1" in feature_columns:
    full_eval_dataset["DKPrice_lag1"] = full_eval_dataset["DKPrice"].shift(1)
missing_in_eval = [c for c in feature_columns if c not in full_eval_dataset.columns]
if missing_in_eval:
    raise ValueError(f"full_eval_dataset is missing feature columns from dataset_train_input: {missing_in_eval}")
# DKPrice must be the first column so get_predictions treats it as the target
full_eval_dataset = full_eval_dataset[["DKPrice", "Time"] + feature_columns].copy()

# Initialize W&B run
try:
    wandb.finish()
except Exception:
    pass
test_run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_TEST_RUN_NAME,
    job_type="evaluation",
    config={
        "price_zone": PRICE_ZONE,
        "test_start": str(TEST_START),
        "test_end": str(TEST_END),
        "test_window": int(TEST_WINDOW),
        "test_rows": int(len(test_set)),
        "model_artifact": WANDB_ARTIFACT_NAME,
        "forecast_horizon": int(FORECAST_HORIZON),
        "predict_period": int(PREDICT_PERIOD),
        "stride": int(STRIDE),
        "prediction_method": "weekly_168h_autoregressive",
    },
    tags=["lstm", "final-model", "test", "evaluation", "weekly-metrics"],
    reinit=True,
    settings=wandb.Settings(start_method="thread"),
)

# Load trained model
model_artifact = test_run.use_artifact(f"{WANDB_ARTIFACT_NAME}:latest")
artifact_dir = Path(model_artifact.download())
model_path = artifact_dir / "model.joblib"
if not model_path.exists():
    raise ValueError(f"Could not find model.joblib in downloaded artifact: {artifact_dir}")

model = joblib.load(model_path)
print(f"Loaded model artifact: {WANDB_ARTIFACT_NAME}:latest")
print(f"Test window: {TEST_START} -> {TEST_END} ({len(test_set)} rows)")
print(f"Feature columns ({len(feature_columns)}): {feature_columns}")

# Fit scaler on the same training tail used during training.
# Compute target-lag columns on the full sorted history before taking the tail
# so that the first row of the tail has a valid lag value.
train_for_scaler = history_source.sort_values("Time").copy()
if "DKPrice_lag1" in feature_columns and "DKPrice_lag1" not in train_for_scaler.columns:
    train_for_scaler["DKPrice_lag1"] = train_for_scaler["DKPrice"].shift(1)
train_for_scaler = train_for_scaler.tail(TRAIN_WINDOW).reset_index(drop=True)
X_scaler = train_for_scaler[feature_columns].astype(np.float32)
scaler = StandardScaler()
scaler.fit(X_scaler)

# Build 2025 weekly folds
folds = _build_validation_folds(
    data=full_eval_dataset,
    val_window=TEST_WINDOW,
    val_start=str(TEST_START),
    predict_period=PREDICT_PERIOD,
    stride=STRIDE,
    validation_reference=test_set,
)
print(f"Generated {len(folds)} weekly folds with predict_period={PREDICT_PERIOD}, stride={STRIDE}.")

# Metric helper
def smape_mean(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.abs(y_true) + np.abs(y_pred)
    vals = np.where(denom == 0, 0.0, 200.0 * np.abs(y_pred - y_true) / denom)
    return float(np.mean(vals))

# Evaluate fold-by-fold (week-by-week)
all_week_evals = []
day_smapes_per_week = {}  # (fold_no, week_no) -> {day_num -> smape}

for fold in folds:
    fold_no = int(fold["fold"])
    fold_val_data = test_set.loc[
        (test_set["Time"] >= fold["val_start"]) & (test_set["Time"] <= fold["val_end"])
    ].copy()

    preds_by_week = get_predictions(
        model=model,
        dataset=full_eval_dataset,
        val_start=fold["val_start"],
        val_end=fold["val_end"],
        forecast_horizon=FORECAST_HORIZON,
        fitted_scaler=scaler,
        dk_zone=PRICE_ZONE,
        rf_models=rf_models if "rf_models" in globals() else None,
        use_precomputed_feature_values=use_precomputed_feature_values if "use_precomputed_feature_values" in globals() else False,
        precomputed_feature_predictions=feature_predictions if "feature_predictions" in globals() else None,
        use_forecasted_history=USE_FORECASTED_HISTORY,
    )

    for week_no, week_pred_df in preds_by_week.items():
        week_eval = week_pred_df.merge(
            fold_val_data[["Time", "DKPrice"]],
            on="Time",
            how="left",
        )
        week_eval = week_eval.dropna(subset=["DKPrice"]).copy()
        if week_eval.empty:
            continue

        week_eval["fold"] = fold_no
        week_eval["week_in_fold"] = int(week_no)
        all_week_evals.append(week_eval)

        week_key = (fold_no, int(week_no))
        day_smapes_per_week[week_key] = {}
        week_eval_sorted = week_eval.sort_values("Time").reset_index(drop=True)

        for day_num in range(1, 8):
            start_hour = (day_num - 1) * 24
            end_hour = day_num * 24
            day_data = week_eval_sorted.iloc[start_hour:end_hour]
            if len(day_data) > 0:
                day_smapes_per_week[week_key][day_num] = smape_mean(
                    day_data["DKPrice"].values,
                    day_data["Prediction"].values,
                )

if not all_week_evals:
    raise RuntimeError("No aligned predictions produced for 2025 weekly folds.")

results_df = pd.concat(all_week_evals, ignore_index=True).sort_values("Time").reset_index(drop=True)
results_df = results_df.rename(columns={"DKPrice": "Actual"})
results_df["Error"] = results_df["Actual"] - results_df["Prediction"]
results_df["Abs_Error"] = np.abs(results_df["Error"])

y_actual = results_df["Actual"].values
y_pred = results_df["Prediction"].values

# Overall metrics
overall_mse = float(np.mean((y_actual - y_pred) ** 2))
overall_smape = smape_mean(y_actual, y_pred)
overall_mae = float(np.mean(np.abs(y_actual - y_pred)))
overall_rmse = float(np.sqrt(overall_mse))

print("\n=== Overall Test Results ===")
print(f"MSE Loss:  {overall_mse:.6f}")
print(f"SMAPE:     {overall_smape:.6f}")
print(f"MAE:       {overall_mae:.6f}")
print(f"RMSE:      {overall_rmse:.6f}")

# Weekly metrics (calendar week)
print("\n=== Weekly Metrics ===")
results_df["Week"] = results_df["Time"].dt.isocalendar().week
weekly_results = []
for week, group in results_df.groupby("Week"):
    week_actual = group["Actual"].values
    week_pred = group["Prediction"].values
    week_rmse = float(np.sqrt(np.mean((week_actual - week_pred) ** 2)))
    week_mae = float(np.mean(np.abs(week_actual - week_pred)))
    week_smape = smape_mean(week_actual, week_pred)
    weekly_results.append({
        "week": int(week),
        "rmse": week_rmse,
        "mae": week_mae,
        "smape": week_smape,
        "n_samples": len(group),
    })
    print(f"Week {week:02d}: RMSE={week_rmse:.4f}, MAE={week_mae:.4f}, SMAPE={week_smape:.4f}")

# Daily metrics
print("\n=== Daily Metrics (by Date) ===")
results_df["Date"] = results_df["Time"].dt.date
daily_results = []
for date, group in results_df.groupby("Date"):
    day_actual = group["Actual"].values
    day_pred = group["Prediction"].values
    day_rmse = float(np.sqrt(np.mean((day_actual - day_pred) ** 2)))
    day_mae = float(np.mean(np.abs(day_actual - day_pred)))
    day_smape = smape_mean(day_actual, day_pred)
    daily_results.append({
        "date": str(date),
        "rmse": day_rmse,
        "mae": day_mae,
        "smape": day_smape,
        "n_samples": len(group),
    })
    print(f"{date}: RMSE={day_rmse:.4f}, MAE={day_mae:.4f}, SMAPE={day_smape:.4f}")

# Day-of-forecast metrics (Day 1..7 inside each predicted week)
print("\n=== Day-of-Forecast Metrics (within each 168-hour period) ===")
day_of_forecast_smapes = {}
for day_num in range(1, 8):
    smapes = [
        day_smapes_per_week[w].get(day_num)
        for w in day_smapes_per_week
        if day_num in day_smapes_per_week[w]
    ]
    smapes = [s for s in smapes if s is not None and not np.isnan(s)]
    day_of_forecast_smapes[day_num] = float(np.mean(smapes)) if smapes else float("nan")
    print(f"Day {day_num} (hours {(day_num-1)*24}-{day_num*24-1}): avg SMAPE={day_of_forecast_smapes[day_num]:.4f}")

# Average weekly/daily metrics
avg_weekly_rmse = float(np.mean([r["rmse"] for r in weekly_results]))
avg_weekly_mae = float(np.mean([r["mae"] for r in weekly_results]))
avg_weekly_smape = float(np.mean([r["smape"] for r in weekly_results]))

avg_daily_rmse = float(np.mean([r["rmse"] for r in daily_results]))
avg_daily_mae = float(np.mean([r["mae"] for r in daily_results]))
avg_daily_smape = float(np.mean([r["smape"] for r in daily_results]))

print("\n=== Average Weekly Metrics ===")
print(f"Avg Weekly RMSE: {avg_weekly_rmse:.6f}")
print(f"Avg Weekly MAE:  {avg_weekly_mae:.6f}")
print(f"Avg Weekly SMAPE: {avg_weekly_smape:.6f}")

print("\n=== Average Daily Metrics ===")
print(f"Avg Daily RMSE: {avg_daily_rmse:.6f}")
print(f"Avg Daily MAE:  {avg_daily_mae:.6f}")
print(f"Avg Daily SMAPE: {avg_daily_smape:.6f}")

# Save all hourly predictions to CSV
output_root = Path(project_root) / "Deep learners" / "LSTM"
output_root.mkdir(parents=True, exist_ok=True)
predictions_csv_path = output_root / f"{WANDB_RUN_NAME}_test_predictions.csv"
results_df.to_csv(predictions_csv_path, index=False, sep=";", decimal=".")
print(f"\nHourly predictions saved to: {predictions_csv_path}")

# Log metrics
test_run.log({
    "test_overall_MSE_loss": overall_mse,
    "test_overall_SMAPE": overall_smape,
    "test_overall_MAE": overall_mae,
    "test_overall_RMSE": overall_rmse,
    "test_avg_weekly_RMSE": avg_weekly_rmse,
    "test_avg_weekly_MAE": avg_weekly_mae,
    "test_avg_weekly_SMAPE": avg_weekly_smape,
    "test_avg_daily_RMSE": avg_daily_rmse,
    "test_avg_daily_MAE": avg_daily_mae,
    "test_avg_daily_SMAPE": avg_daily_smape,
    "test_avg_smape_day_1": day_of_forecast_smapes.get(1, float("nan")),
    "test_avg_smape_day_2": day_of_forecast_smapes.get(2, float("nan")),
    "test_avg_smape_day_3": day_of_forecast_smapes.get(3, float("nan")),
    "test_avg_smape_day_4": day_of_forecast_smapes.get(4, float("nan")),
    "test_avg_smape_day_5": day_of_forecast_smapes.get(5, float("nan")),
    "test_avg_smape_day_6": day_of_forecast_smapes.get(6, float("nan")),
    "test_avg_smape_day_7": day_of_forecast_smapes.get(7, float("nan")),
})

weekly_df = pd.DataFrame(weekly_results)
daily_df = pd.DataFrame(daily_results)
test_run.log({
    "test_weekly_metrics": wandb.Table(dataframe=weekly_df),
    "test_daily_metrics": wandb.Table(dataframe=daily_df),
    "test_predictions_sample": wandb.Table(dataframe=results_df.head(100)),
})

test_run.summary.update({
    "test_overall_MSE_loss": overall_mse,
    "test_overall_SMAPE": overall_smape,
    "test_overall_MAE": overall_mae,
    "test_overall_RMSE": overall_rmse,
    "test_avg_weekly_RMSE": avg_weekly_rmse,
    "test_avg_weekly_MAE": avg_weekly_mae,
    "test_avg_weekly_SMAPE": avg_weekly_smape,
    "test_avg_daily_RMSE": avg_daily_rmse,
    "test_avg_daily_MAE": avg_daily_mae,
    "test_avg_daily_SMAPE": avg_daily_smape,
    "test_samples": int(len(results_df)),
    "test_period_start": str(TEST_START),
    "test_period_end": str(TEST_END),
    "n_weeks": len(weekly_results),
    "n_days": len(daily_results),
})


print(f"\nTest predictions and metrics logged to W&B run: {WANDB_TEST_RUN_NAME}")
wandb.finish()


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.base import BaseEstimator, RegressorMixin
from torch.utils.data import DataLoader, Dataset


def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def smape_mean(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.abs(y_true) + np.abs(y_pred)
    vals = np.where(denom == 0, 0.0, 200.0 * np.abs(y_pred - y_true) / denom)
    return float(np.mean(vals))


# ---------------------------------------------------------------------------
# Seq2Seq Dataset
# ---------------------------------------------------------------------------

class Seq2SeqDataset(Dataset):
    """
    Builds encoder/decoder/target triplets on-the-fly for seq2seq training.

    For each valid index i in [seq_len, n_samples - horizon):
      encoder_input  = concat(X[i-seq_len:i], y[i-seq_len:i].reshape(-1,1))
                       shape (seq_len, n_features + 1)  – scaled features + unscaled DKPrice
      decoder_input  = X[i : i+horizon]
                       shape (horizon, n_features)       – scaled future features
      target         = y[i : i+horizon]
                       shape (horizon,)                  – future DKPrice values
    """

    def __init__(self, X_np: np.ndarray, y_np: np.ndarray, seq_len: int, horizon: int):
        self.X = X_np
        self.y = y_np
        self.seq_len = seq_len
        self.horizon = horizon
        self.n_valid = max(0, len(X_np) - seq_len - horizon + 1)

    def __len__(self) -> int:
        return self.n_valid

    def __getitem__(self, idx: int):
        i = idx + self.seq_len
        enc_x = self.X[i - self.seq_len : i]                          # (seq_len, n_features)
        enc_price = self.y[i - self.seq_len : i].reshape(-1, 1)       # (seq_len, 1)
        encoder_input = np.concatenate([enc_x, enc_price], axis=1)    # (seq_len, n_features+1)
        decoder_input = self.X[i : i + self.horizon]                  # (horizon, n_features)
        target = self.y[i : i + self.horizon]                         # (horizon,)
        return (
            torch.tensor(encoder_input, dtype=torch.float32),
            torch.tensor(decoder_input, dtype=torch.float32),
            torch.tensor(target, dtype=torch.float32),
        )


# ---------------------------------------------------------------------------
# LSTM Autoencoder module
# ---------------------------------------------------------------------------

class LSTMAutoencoder(nn.Module):
    """
    Seq2seq LSTM Autoencoder for multi-step price forecasting.

    Architecture (based on Option 1 in the design document):
      1. Feature encoder  – a small dense MLP applied per timestep that
                            compresses (n_features + 1) inputs to latent_dim.
      2. Encoder LSTM     – processes the latent sequence and produces a
                            compressed hidden state.
      3. Decoder LSTM     – initialised with the encoder's final hidden state
                            and fed future feature forecasts; outputs
                            decoder_horizon price predictions.

    Parameters
    ----------
    encoder_input_size : int
        Number of encoder input features per timestep (n_features + 1 for DKPrice).
    decoder_input_size : int
        Number of decoder input features per timestep (n_features, no DKPrice).
    latent_dim : int
        Output dimension of the per-timestep feature encoder.
    encoder_hidden_size : int
        Hidden size of the encoder LSTM.
    decoder_hidden_size : int
        Hidden size of the decoder LSTM.
    layers : int
        Number of LSTM layers (shared between encoder and decoder).
    dense_layers : int
        Depth of the per-timestep feature encoder MLP (>=1).
    dropout : float
        Shared fallback dropout. Used for dense feature encoder dropout and as
        fallback for encoder/decoder LSTM dropout when specific values are not set.
    encoder_dropout : float or None
        Encoder LSTM dropout (active when encoder_layers > 1). If None, uses dropout.
    decoder_dropout : float or None
        Decoder LSTM dropout (active when decoder_layers > 1). If None, uses dropout.
    """

    def __init__(
        self,
        encoder_input_size: int,
        decoder_input_size: int,
        latent_dim: int = 16,
        encoder_hidden_size: int = 64,
        decoder_hidden_size: int = 64,
        layers: int = 1,
        encoder_layers=None,
        decoder_layers=None,
        dense_layers: int = 1,
        dropout: float = 0.0,
        encoder_dropout=None,
        decoder_dropout=None,
    ):
        super().__init__()
        if encoder_layers is None:
            encoder_layers = layers
        if decoder_layers is None:
            decoder_layers = layers
        self.encoder_num_layers = encoder_layers
        self.decoder_num_layers = decoder_layers
        encoder_layers = int(encoder_layers)
        decoder_layers = int(decoder_layers)

        if encoder_dropout is None:
            encoder_dropout = dropout
        if decoder_dropout is None:
            decoder_dropout = dropout
        encoder_dropout = float(encoder_dropout)
        decoder_dropout = float(decoder_dropout)

        # --- Feature encoder (dense MLP applied per timestep) ---
        enc_layers = []
        input_size = encoder_input_size

        insz_latentdim_diff = input_size - latent_dim
        change_per_layer = round(insz_latentdim_diff / dense_layers)

        in_sizes = [input_size]

        for _ in range(dense_layers - 1):
            next_in_sz = in_sizes[-1] - change_per_layer
            if next_in_sz < latent_dim:
                next_in_sz = latent_dim
            in_sizes.append(next_in_sz)

        in_sizes.append(latent_dim)
        
        for sz in range(len(in_sizes) - 1):
            in_sz = in_sizes[sz]
            out_sz = in_sizes[sz + 1]

            enc_layers.append(nn.Linear(in_sz, out_sz))

            if sz < len(in_sizes) - 2:
                enc_layers.append(nn.ReLU())
                if dropout > 0.0:
                    enc_layers.append(nn.Dropout(dropout))

        self.feature_encoder = nn.Sequential(*enc_layers)

        enc_lstm_dropout = encoder_dropout if encoder_layers > 1 else 0.0
        dec_lstm_dropout = decoder_dropout if decoder_layers > 1 else 0.0

        # --- Encoder LSTM ---
        self.encoder_lstm = nn.LSTM(
            input_size=latent_dim,
            hidden_size=encoder_hidden_size,
            num_layers=encoder_layers,
            dropout=enc_lstm_dropout,
            batch_first=True,
        )

        # --- Optional hidden-state adapter ---
        self.needs_adapter = (encoder_hidden_size != decoder_hidden_size)
        if self.needs_adapter:
            self.hidden_adapter = nn.Linear(encoder_hidden_size, decoder_hidden_size)

        # --- Decoder LSTM ---
        self.decoder_lstm = nn.LSTM(
            input_size=decoder_input_size,
            hidden_size=decoder_hidden_size,
            num_layers=decoder_layers,
            dropout=dec_lstm_dropout,
            batch_first=True,
        )

        # --- Output projection ---
        self.fc = nn.Linear(decoder_hidden_size, 1)

    def _match_decoder_layers(self, h: torch.Tensor, c: torch.Tensor):
        """Match encoder state depth to decoder num_layers."""
        enc_layers = h.size(0)
        dec_layers = self.decoder_num_layers
        if enc_layers == dec_layers:
            return h, c

        if dec_layers < enc_layers:
            # Keep the top-most encoder layers for decoder initialization.
            return h[-dec_layers:], c[-dec_layers:]

        # dec_layers > enc_layers: pad with copies of the top encoder layer.
        pad = dec_layers - enc_layers
        h_pad = h[-1:].repeat(pad, 1, 1)
        c_pad = c[-1:].repeat(pad, 1, 1)
        return torch.cat([h, h_pad], dim=0), torch.cat([c, c_pad], dim=0)

    def forward(self, encoder_x: torch.Tensor, decoder_x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        encoder_x : Tensor of shape (batch, seq_len, encoder_input_size)
        decoder_x : Tensor of shape (batch, horizon, decoder_input_size)

        Returns
        -------
        Tensor of shape (batch, horizon)
        """
        # Feature encoding applied identically to every timestep
        latent = self.feature_encoder(encoder_x)          # (batch, seq_len, latent_dim)

        # Encoder LSTM – only final hidden state is used
        _, (h_enc, c_enc) = self.encoder_lstm(latent)     # (layers, batch, enc_hidden)

        # Adapt hidden state dimensions if encoder/decoder sizes differ
        if self.needs_adapter:
            h_dec = self.hidden_adapter(h_enc)
            c_dec = self.hidden_adapter(c_enc)
        else:
            h_dec, c_dec = h_enc, c_enc

        # Match layer depth when encoder_layers != decoder_layers.
        h_dec, c_dec = self._match_decoder_layers(h_dec, c_dec)

        # Decoder LSTM initialized with encoder's final state
        out, _ = self.decoder_lstm(decoder_x, (h_dec, c_dec))  # (batch, horizon, dec_hidden)

        return self.fc(out).squeeze(-1)                    # (batch, horizon)


# ---------------------------------------------------------------------------
# Scikit-learn compatible regressor
# ---------------------------------------------------------------------------

class TorchLSTMAERegressor(BaseEstimator, RegressorMixin):
    """
    Scikit-learn style regressor wrapping LSTMAutoencoder.

    fit(X, y)
        X : (n_samples, n_features) – pre-scaled feature matrix (DKPrice excluded).
        y : (n_samples,)            – raw DKPrice target values.
        Internally builds Seq2SeqDataset and trains the seq2seq model.

    predict_ae(encoder_x, decoder_x)
        Used by week_predictions2_AE.get_predictions() for block inference.
        encoder_x : (1, seq_len, n_features+1) – scaled features + unscaled DKPrice
        decoder_x : (1, horizon, n_features)   – scaled future features
        Returns   : (horizon,) predicted prices.

    predict(X)
        2D fallback interface for SHAP / sklearn compatibility.
        Each row of X is broadcast to a constant 168-step decoder sequence;
        a zero encoder input is used as neutral baseline.
        Returns the mean prediction across the horizon.
    """

    def __init__(
        self,
        latent_dim: int = 16,
        encoder_hidden_size: int = 64,
        decoder_hidden_size: int = 64,
        layers: int = 1,
        encoder_layers=None,
        decoder_layers=None,
        dense_layers: int = 1,
        learning_rate: float = 1e-3,
        epochs: int = 40,
        batch_size: int = 32,
        sequence_length: int = 168,
        decoder_horizon: int = 168,
        dropout: float = 0.0,
        encoder_dropout=None,
        decoder_dropout=None,
        random_state: int = 42,
        log_epoch_metrics: bool = False,
        log_prefix: str = "",
        warm_start: bool = False,
    ):
        self.latent_dim = latent_dim
        self.encoder_hidden_size = encoder_hidden_size
        self.decoder_hidden_size = decoder_hidden_size
        self.layers = layers
        self.encoder_layers = int(encoder_layers) if encoder_layers is not None else int(layers)
        self.decoder_layers = int(decoder_layers) if decoder_layers is not None else int(layers)
        self.dense_layers = dense_layers
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.sequence_length = sequence_length
        self.decoder_horizon = decoder_horizon
        self.dropout = dropout
        self.encoder_dropout = float(encoder_dropout) if encoder_dropout is not None else float(dropout)
        self.decoder_dropout = float(decoder_dropout) if decoder_dropout is not None else float(dropout)
        self.random_state = random_state
        self.log_epoch_metrics = log_epoch_metrics
        self.log_prefix = log_prefix
        self.warm_start = warm_start

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _initialize_model_state(self, encoder_input_size: int, decoder_input_size: int):
        self.device_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.pin_memory_ = self.device_.type == "cuda"
        self.encoder_input_size_ = int(encoder_input_size)
        self.decoder_input_size_ = int(decoder_input_size)
        self.model_ = LSTMAutoencoder(
            encoder_input_size=self.encoder_input_size_,
            decoder_input_size=self.decoder_input_size_,
            latent_dim=int(self.latent_dim),
            encoder_hidden_size=int(self.encoder_hidden_size),
            decoder_hidden_size=int(self.decoder_hidden_size),
            layers=int(self.layers),
            encoder_layers=int(self.encoder_layers),
            decoder_layers=int(self.decoder_layers),
            dense_layers=int(self.dense_layers),
            dropout=float(self.dropout),
            encoder_dropout=float(self.encoder_dropout),
            decoder_dropout=float(self.decoder_dropout),
        ).to(self.device_)

        # Load pretrained feature encoder weights from Stage 1 (if set via
        # set_pretrained_encoder()).  Encoder params are frozen when freeze=True,
        # so the optimizer below excludes them automatically.
        _sd = getattr(self, "_pretrained_encoder_state_dict_", None)
        if _sd is not None:
            self.model_.feature_encoder.load_state_dict(_sd)
            if getattr(self, "_freeze_encoder_", True):
                for p in self.model_.feature_encoder.parameters():
                    p.requires_grad = False

        self.loss_fn_ = nn.MSELoss()
        # Only pass parameters that require gradients so frozen encoder layers
        # are not updated even if the optimizer sees them.
        self.optimizer_ = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.model_.parameters()),
            lr=float(self.learning_rate),
        )
        self.epoch_losses_ = []
        self.epoch_smapes_ = []
        self.epoch_maes_ = []
        self.epoch_rmses_ = []
        self._epochs_trained_ = 0

    # ------------------------------------------------------------------
    # set_pretrained_encoder
    # ------------------------------------------------------------------

    def set_pretrained_encoder(
        self, state_dict: dict, freeze: bool = True
    ) -> "TorchLSTMAERegressor":
        """
        Store pretrained feature encoder weights to be loaded at the next model
        initialisation (i.e. the first fit() call, or when warm_start=False).

        Call this before the first fit() / run_cross_validation() call so the
        weights are in place when _initialize_model_state() creates model_.

        Parameters
        ----------
        state_dict : OrderedDict returned by FeatureAutoencoder.encoder.state_dict()
                     (i.e. ``result["best_encoder_state_dict"]`` from Stage 1).
        freeze     : if True the feature_encoder parameters are frozen so only
                     the encoder/decoder LSTM and output layers are updated
                     during Stage 2 training.
        """
        import copy as _copy
        self._pretrained_encoder_state_dict_ = _copy.deepcopy(state_dict)
        self._freeze_encoder_ = freeze
        return self

    # ------------------------------------------------------------------
    # fit
    # ------------------------------------------------------------------

    def fit(self, X, y):
        set_seed(self.random_state)

        X_np = np.asarray(X, dtype=np.float32)
        y_np = np.asarray(y, dtype=np.float32).reshape(-1)

        if X_np.ndim != 2:
            raise ValueError(f"Expected 2-D X, got shape {X_np.shape}.")

        n_decoder_features = X_np.shape[1]
        n_encoder_features = n_decoder_features + 1   # features + DKPrice
        seq_len = int(self.sequence_length)
        horizon = int(self.decoder_horizon)

        # Build or reuse the dataset (reused across warm-start epochs for speed)
        needs_rebuild = (
            not hasattr(self, "_train_dataset_")
            or getattr(self, "_train_X_shape_", None) != X_np.shape
            or getattr(self, "_train_y_len_", None) != len(y_np)
        )
        if needs_rebuild:
            self._train_dataset_ = Seq2SeqDataset(X_np, y_np, seq_len, horizon)
            self._train_X_shape_ = X_np.shape
            self._train_y_len_ = len(y_np)

        if len(self._train_dataset_) == 0:
            raise ValueError(
                f"Training set too small to build any seq2seq sample "
                f"(need > seq_len+horizon={seq_len+horizon} rows, got {len(X_np)})."
            )

        # Initialise or reuse the model
        needs_reinit = (
            (not bool(self.warm_start))
            or (not hasattr(self, "model_"))
            or (not hasattr(self, "encoder_input_size_"))
            or (int(self.encoder_input_size_) != n_encoder_features)
            or (int(self.decoder_input_size_) != n_decoder_features)
        )
        if needs_reinit:
            self._initialize_model_state(
                encoder_input_size=n_encoder_features,
                decoder_input_size=n_decoder_features,
            )
        elif not hasattr(self, "device_"):
            self.device_ = next(self.model_.parameters()).device
            self.pin_memory_ = self.device_.type == "cuda"

        loader = DataLoader(
            self._train_dataset_,
            batch_size=int(self.batch_size),
            shuffle=True,
            pin_memory=self.pin_memory_,
        )

        self.model_.train()
        for _ in range(int(self.epochs)):
            batch_losses = []
            epoch_preds = []
            epoch_targets = []

            for enc_batch, dec_batch, tgt_batch in loader:
                enc_batch = enc_batch.to(self.device_, non_blocking=self.pin_memory_)
                dec_batch = dec_batch.to(self.device_, non_blocking=self.pin_memory_)
                tgt_batch = tgt_batch.to(self.device_, non_blocking=self.pin_memory_)

                self.optimizer_.zero_grad()
                preds = self.model_(enc_batch, dec_batch)   # (batch, horizon)
                loss = self.loss_fn_(preds, tgt_batch)
                loss.backward()
                self.optimizer_.step()

                batch_losses.append(float(loss.item()))
                epoch_preds.append(preds.detach().cpu().numpy().reshape(-1))
                epoch_targets.append(tgt_batch.detach().cpu().numpy().reshape(-1))

            epoch_loss = float(np.mean(batch_losses)) if batch_losses else float("nan")
            self.epoch_losses_.append(epoch_loss)

            if epoch_preds and epoch_targets:
                y_pred_epoch = np.concatenate(epoch_preds)
                y_true_epoch = np.concatenate(epoch_targets)
                epoch_smape = smape_mean(y_true_epoch, y_pred_epoch)
                epoch_mae = float(np.mean(np.abs(y_true_epoch - y_pred_epoch)))
                epoch_rmse = float(np.sqrt(np.mean((y_true_epoch - y_pred_epoch) ** 2)))
            else:
                epoch_smape = epoch_mae = epoch_rmse = float("nan")

            self.epoch_smapes_.append(float(epoch_smape))
            self.epoch_maes_.append(float(epoch_mae))
            self.epoch_rmses_.append(float(epoch_rmse))
            self._epochs_trained_ += 1

            if bool(self.log_epoch_metrics):
                try:
                    import wandb
                    if wandb.run is not None:
                        pfx = self.log_prefix
                        wandb.log({
                            f"{pfx}train_MSE_loss" if pfx else "train_MSE_loss": epoch_loss,
                            f"{pfx}train_smape"    if pfx else "train_smape":    float(epoch_smape),
                            f"{pfx}train_mae"      if pfx else "train_mae":      float(epoch_mae),
                            f"{pfx}train_rmse"     if pfx else "train_rmse":     float(epoch_rmse),
                            f"{pfx}epoch"          if pfx else "epoch":          int(self._epochs_trained_),
                        })
                except Exception:
                    pass

        return self

    # ------------------------------------------------------------------
    # predict_ae  (primary inference path used by week_predictions2_AE)
    # ------------------------------------------------------------------

    def predict_ae(self, encoder_x: np.ndarray, decoder_x: np.ndarray) -> np.ndarray:
        """
        Predict one 168-hour block in a single forward pass.

        Parameters
        ----------
        encoder_x : ndarray of shape (1, seq_len, n_features+1)
            Scaled historical features concatenated with unscaled DKPrice.
        decoder_x : ndarray of shape (1, horizon, n_features)
            Scaled future feature forecasts.

        Returns
        -------
        ndarray of shape (horizon,)
        """
        self.model_.eval()
        enc_t = torch.tensor(encoder_x, dtype=torch.float32).to(self.device_)
        dec_t = torch.tensor(decoder_x, dtype=torch.float32).to(self.device_)
        with torch.no_grad():
            out = self.model_(enc_t, dec_t)    # (1, horizon)
        return out.squeeze(0).detach().cpu().numpy()

    # ------------------------------------------------------------------
    # predict  (2-D fallback for SHAP / sklearn compatibility)
    # ------------------------------------------------------------------

    def predict(self, X) -> np.ndarray:
        """
        2-D input interface for SHAP / sklearn compatibility.

        Each row of X is broadcast to a constant `decoder_horizon`-step decoder
        sequence.  A zero encoder input is used as a neutral baseline.
        Returns the mean prediction across the horizon for each sample.
        """
        X_np = np.asarray(X, dtype=np.float32)
        if X_np.ndim != 2:
            raise ValueError(
                f"predict() expects 2-D X; got shape {X_np.shape}. "
                "For block inference use predict_ae(encoder_x, decoder_x)."
            )

        n_samples, n_features = X_np.shape
        horizon = int(self.decoder_horizon)
        seq_len = int(self.sequence_length)

        # Neutral encoder (all zeros)
        enc_np = np.zeros((1, seq_len, n_features + 1), dtype=np.float32)
        enc_t = torch.tensor(enc_np, dtype=torch.float32).to(self.device_)

        self.model_.eval()
        preds = []
        with torch.no_grad():
            for i in range(n_samples):
                # Replicate single feature row across the full decoder horizon
                dec_np = np.tile(X_np[i : i + 1], (horizon, 1))[np.newaxis, :]  # (1, horizon, n)
                dec_t = torch.tensor(dec_np, dtype=torch.float32).to(self.device_)
                out = self.model_(enc_t, dec_t)   # (1, horizon)
                preds.append(float(out.mean().item()))

        return np.array(preds, dtype=np.float32)


In [ ]:
import copy
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset


# ---------------------------------------------------------------------------
# Differentiable SMAPE loss
# ---------------------------------------------------------------------------

class SMAPELoss(nn.Module):
    """SMAPE loss: 200 * |y_pred - y_true| / (|y_true| + |y_pred| + eps)."""

    def __init__(self, eps: float = 1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        denom = torch.abs(y_true) + torch.abs(y_pred) + self.eps
        return torch.mean(200.0 * torch.abs(y_pred - y_true) / denom)


# ---------------------------------------------------------------------------
# Feature Autoencoder – mirrors the architecture of LSTMAutoencoder.feature_encoder
# ---------------------------------------------------------------------------

class FeatureAutoencoder(nn.Module):
    """
    Standalone autoencoder for tuning the per-timestep feature encoder MLP.

    Encoder architecture is identical to LSTMAutoencoder.feature_encoder:
      dense_layers=1 → single Linear(input_size → latent_dim)
      dense_layers=k → k-1 × [Linear → ReLU → (Dropout)] + final Linear → latent_dim

    Decoder mirrors the encoder (symmetric MLP from latent_dim back to input_size).

    Parameters
    ----------
    input_size  : number of features per timestep (n_features + 1 for DKPrice, same
                  as encoder_input_size in LSTMAutoencoder)
    latent_dim  : output dimension of the encoder (hyperparameter to tune)
    dense_layers: depth of the encoder/decoder MLP (≥1)
    dropout     : dropout rate applied between hidden layers (only if dense_layers > 1)
    """

    def __init__(
        self,
        input_size: int,
        latent_dim: int,
        dense_layers: int,
        dropout: float = 0.0,
    ):
        super().__init__()

        # ---- Encoder (same as LSTMAutoencoder.feature_encoder) ----
        enc = []

        insz_latentdim_diff = input_size - latent_dim
        change_per_layer = round(insz_latentdim_diff / dense_layers)

        in_sizes = [input_size]

        for _ in range(dense_layers - 1):
            next_in_sz = in_sizes[-1] - change_per_layer
            if next_in_sz < latent_dim:
                next_in_sz = latent_dim
            in_sizes.append(next_in_sz)

        in_sizes.append(latent_dim)
        
        for sz in range(len(in_sizes) - 1):
            in_sz = in_sizes[sz]
            out_sz = in_sizes[sz + 1]

            enc.append(nn.Linear(in_sz, out_sz))

            if sz < len(in_sizes) - 2:
                enc.append(nn.ReLU())
                if dropout > 0.0:
                    enc.append(nn.Dropout(dropout))

        self.encoder = nn.Sequential(*enc)

        # ---- Decoder (symmetric to encoder) ----
        dec = []

        decoder_sizes = list(reversed(in_sizes))

        for sz in range(len(decoder_sizes) - 1):
            in_sz = decoder_sizes[sz]
            out_sz = decoder_sizes[sz + 1]

            dec.append(nn.Linear(in_sz, out_sz))

            if sz < len(decoder_sizes) - 2:
                dec.append(nn.ReLU())
                if dropout > 0.0:
                    dec.append(nn.Dropout(dropout))

        self.decoder = nn.Sequential(*dec)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)


# ---------------------------------------------------------------------------
# Training helper
# ---------------------------------------------------------------------------

def train_feature_autoencoder(
    X_train: np.ndarray,
    X_val: np.ndarray,
    latent_dim: int,
    dense_layers: int,
    learning_rate: float,
    max_epochs: int,
    patience: int,
    batch_size: int,
    dropout: float = 0.0,
    random_state: int = 42,
    log_wandb: bool = False,
) -> dict:
    """
    Train a FeatureAutoencoder with SMAPE reconstruction loss.

    Parameters
    ----------
    X_train / X_val : float32 arrays of shape (n_samples, input_size).
                      Should be the scaled feature matrix concatenated with
                      raw DKPrice, matching the encoder input in Seq2SeqDataset.
    log_wandb       : whether to call wandb.log() per epoch.

    Returns
    -------
    dict with keys: best_val_smape, best_epoch, epochs_trained, epoch_history,
                    best_encoder_state_dict (state_dict of the encoder at best epoch,
                    ready to be loaded into LSTMAutoencoder.feature_encoder).
    """
    set_seed(random_state)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pin_memory = device.type == "cuda"

    input_size = X_train.shape[1]
    model = FeatureAutoencoder(
        input_size=input_size,
        latent_dim=latent_dim,
        dense_layers=dense_layers,
        dropout=dropout,
    ).to(device)

    criterion = SMAPELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    train_t = torch.tensor(X_train, dtype=torch.float32)
    val_t = torch.tensor(X_val, dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(train_t),
        batch_size=batch_size,
        shuffle=True,
        pin_memory=pin_memory,
    )

    best_val_smape = float("inf")
    best_epoch = 0
    patience_counter = 0
    epoch_history = []
    epoch = 0
    best_encoder_state_dict = None   # saved at the epoch with lowest val SMAPE

    for epoch in range(1, max_epochs + 1):
        model.train()
        batch_losses = []
        for (x_batch,) in loader:
            x_batch = x_batch.to(device, non_blocking=pin_memory)
            optimizer.zero_grad()
            recon = model(x_batch)
            loss = criterion(recon, x_batch)
            loss.backward()
            optimizer.step()
            batch_losses.append(float(loss.item()))

        train_smape = float(np.mean(batch_losses)) if batch_losses else float("nan")

        model.eval()
        with torch.no_grad():
            val_recon = model(val_t.to(device))
            val_smape = float(criterion(val_recon, val_t.to(device)).item())

        epoch_history.append({"epoch": epoch, "train_smape": train_smape, "val_smape": val_smape})

        if log_wandb:
            try:
                import wandb as _wandb
                _wandb.log({"epoch": epoch, "train_smape": train_smape, "val_smape": val_smape})
            except Exception:
                pass

        if val_smape < best_val_smape:
            best_val_smape = val_smape
            best_epoch = epoch
            patience_counter = 0
            # Snapshot the encoder weights at this best epoch
            best_encoder_state_dict = copy.deepcopy(model.encoder.state_dict())
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return {
        "best_val_smape": best_val_smape,
        "best_epoch": best_epoch,
        "epochs_trained": epoch,
        "epoch_history": epoch_history,
        "best_encoder_state_dict": best_encoder_state_dict,
    }


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import wandb
import joblib
from sklearn.preprocessing import StandardScaler
from Modules.Validation3_AE import _build_validation_folds
from Modules.week_predictions2_AE_noise import get_predictions

# =====================================================================================
# Test final LSTM Autoencoder on 2025 only, week-by-week (168h blocks)
# =====================================================================================
WANDB_PROJECT = "LSTM_AE_final"
WANDB_RUN_NAME = "DK1_LSTM_AE_2y_2024incl_Lag1_incl_lags_excl_4valFE_lays1_FE_LatDim33_EnHid28_DeHid48_EnDeLays2"  # Must match final training run name
WANDB_TEST_RUN_NAME = f"LSTM_AE_noise_test"
WANDB_ARTIFACT_NAME = f"{WANDB_RUN_NAME}_model"

FORECAST_HORIZON = 168  # 1 week
PREDICT_PERIOD = 8760    # one week per validation fold
STRIDE = 168            # next fold starts next week
TEST_START = pd.Timestamp("2025-01-01 00:00:00")
TEST_WINDOW = 8760
TEST_END = TEST_START + pd.Timedelta(hours=TEST_WINDOW - 1)

# Retrieve zone data
if PRICE_ZONE == "DK1":
    test_source = DK1_test.copy() if "DK1_test" in globals() else None
    history_source = DK1_train.copy() if "DK1_train" in globals() else None
elif PRICE_ZONE == "DK2":
    test_source = DK2_test.copy() if "DK2_test" in globals() else None
    history_source = DK2_train.copy() if "DK2_train" in globals() else None
else:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")

if test_source is None or history_source is None:
    raise ValueError(f"Missing train/test source data for {PRICE_ZONE}. Run data loading first.")

test_source = test_source.sort_values("Time").reset_index(drop=True)
history_source = history_source.sort_values("Time").reset_index(drop=True)

# Test only on 2025 window
test_set = test_source.loc[
    (test_source["Time"] >= TEST_START) & (test_source["Time"] <= TEST_END)
].copy().sort_values("Time").reset_index(drop=True)

if len(test_set) != TEST_WINDOW:
    raise ValueError(
        f"Expected exactly TEST_WINDOW={TEST_WINDOW} rows from {TEST_START} to {TEST_END}, got {len(test_set)}."
    )

# Feature columns: all columns except Time and DKPrice.
# DKPrice is fed directly into the encoder; it is not a feature column.
feature_columns = [c for c in dataset_train_input.columns if c not in ["Time", "DKPrice"]]
if not feature_columns:
    raise ValueError("No feature columns found in dataset_train_input. Run the load data cell first.")

# Build full eval dataset: include end-of-2024 history so the encoder can look back
# seq_len hours before the first 2025 prediction block.
full_eval_dataset = (
    pd.concat([history_source, test_set], ignore_index=True)
    .sort_values("Time")
    .drop_duplicates(subset=["Time"], keep="last")
    .reset_index(drop=True)
)

# DKPrice_lag1 is derived from Price_lag1 during training (INCLUDE_PRICE_LAG1_AS_INPUT).
# The raw source DataFrames store it as Price_lag1, so map it here if needed.
if INCLUDE_PRICE_LAG1_AS_INPUT and "DKPrice_lag1" not in full_eval_dataset.columns:
    if "Price_lag1" in full_eval_dataset.columns:
        full_eval_dataset["DKPrice_lag1"] = full_eval_dataset["Price_lag1"]
    else:
        full_eval_dataset["DKPrice_lag1"] = full_eval_dataset["DKPrice"].shift(1)

missing_in_eval = [c for c in feature_columns if c not in full_eval_dataset.columns]
if missing_in_eval:
    raise ValueError(f"full_eval_dataset is missing feature columns from dataset_train_input: {missing_in_eval}")

# DKPrice must be the first column so get_predictions treats it as the target
full_eval_dataset = full_eval_dataset[["DKPrice", "Time"] + feature_columns].copy()

# Initialize W&B run
try:
    wandb.finish()
except Exception:
    pass
test_run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_TEST_RUN_NAME,
    job_type="evaluation",
    config={
        "price_zone": PRICE_ZONE,
        "test_start": str(TEST_START),
        "test_end": str(TEST_END),
        "test_window": int(TEST_WINDOW),
        "test_rows": int(len(test_set)),
        "model_artifact": WANDB_ARTIFACT_NAME,
        "forecast_horizon": int(FORECAST_HORIZON),
        "predict_period": int(PREDICT_PERIOD),
        "stride": int(STRIDE),
        "prediction_method": "weekly_168h_seq2seq",
    },
    tags=["lstm-ae", "final-model", "test", "evaluation", "weekly-metrics"],
    reinit=True,
    settings=wandb.Settings(start_method="thread"),
)

# Load trained model
model_artifact = test_run.use_artifact(f"{WANDB_ARTIFACT_NAME}:latest")
artifact_dir = Path(model_artifact.download())
model_path = artifact_dir / "model.joblib"
if not model_path.exists():
    raise ValueError(f"Could not find model.joblib in downloaded artifact: {artifact_dir}")

model = joblib.load(model_path)
print(f"Loaded model artifact: {WANDB_ARTIFACT_NAME}:latest")
print(f"Test window: {TEST_START} -> {TEST_END} ({len(test_set)} rows)")
print(f"Feature columns (decoder, {len(feature_columns)}): {feature_columns}")

# Fit scaler on the same training tail used during training.
train_for_scaler = history_source.sort_values("Time").copy()
train_for_scaler = train_for_scaler.tail(TRAIN_WINDOW).reset_index(drop=True)
# DKPrice_lag1 is stored as Price_lag1 in raw source DataFrames; rename to match feature_columns.
if "DKPrice_lag1" not in train_for_scaler.columns and "Price_lag1" in train_for_scaler.columns:
    train_for_scaler["DKPrice_lag1"] = train_for_scaler["Price_lag1"]
X_scaler = train_for_scaler[feature_columns].astype(np.float32)
scaler = StandardScaler()
scaler.fit(X_scaler)

# Build 2025 weekly folds
folds = _build_validation_folds(
    data=full_eval_dataset,
    val_window=TEST_WINDOW,
    val_start=str(TEST_START),
    predict_period=PREDICT_PERIOD,
    stride=STRIDE,
    validation_reference=test_set,
)
print(f"Generated {len(folds)} weekly folds with predict_period={PREDICT_PERIOD}, stride={STRIDE}.")

# Metric helper
def smape_mean(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.abs(y_true) + np.abs(y_pred)
    vals = np.where(denom == 0, 0.0, 200.0 * np.abs(y_pred - y_true) / denom)
    return float(np.mean(vals))

# Evaluate fold-by-fold (week-by-week)
all_week_evals = []
day_smapes_per_week = {}  # (fold_no, week_no) -> {day_num -> smape}

for fold in folds:
    fold_no = int(fold["fold"])
    fold_val_data = test_set.loc[
        (test_set["Time"] >= fold["val_start"]) & (test_set["Time"] <= fold["val_end"])
    ].copy()

    preds_by_week = get_predictions(
        model=model,
        dataset=full_eval_dataset,
        val_start=fold["val_start"],
        val_end=fold["val_end"],
        forecast_horizon=FORECAST_HORIZON,
        fitted_scaler=scaler,
        dk_zone=PRICE_ZONE,
        rf_models=rf_models if "rf_models" in globals() else None,
        use_precomputed_feature_values=use_precomputed_feature_values if "use_precomputed_feature_values" in globals() else False,
        precomputed_feature_predictions=feature_predictions if "feature_predictions" in globals() else None,
        use_forecasted_history=USE_FORECASTED_HISTORY,
    )

    for week_no, week_pred_df in preds_by_week.items():
        week_eval = week_pred_df.merge(
            fold_val_data[["Time", "DKPrice"]],
            on="Time",
            how="left",
        )
        week_eval = week_eval.dropna(subset=["DKPrice"]).copy()
        if week_eval.empty:
            continue

        week_eval["fold"] = fold_no
        week_eval["week_in_fold"] = int(week_no)
        all_week_evals.append(week_eval)

        week_key = (fold_no, int(week_no))
        day_smapes_per_week[week_key] = {}
        week_eval_sorted = week_eval.sort_values("Time").reset_index(drop=True)

        for day_num in range(1, 8):
            start_hour = (day_num - 1) * 24
            end_hour = day_num * 24
            day_data = week_eval_sorted.iloc[start_hour:end_hour]
            if len(day_data) > 0:
                day_smapes_per_week[week_key][day_num] = smape_mean(
                    day_data["DKPrice"].values,
                    day_data["Prediction"].values,
                )

if not all_week_evals:
    raise RuntimeError("No aligned predictions produced for 2025 weekly folds.")

results_df = pd.concat(all_week_evals, ignore_index=True).sort_values("Time").reset_index(drop=True)
results_df = results_df.rename(columns={"DKPrice": "Actual"})
results_df["Error"] = results_df["Actual"] - results_df["Prediction"]
results_df["Abs_Error"] = np.abs(results_df["Error"])

y_actual = results_df["Actual"].values
y_pred = results_df["Prediction"].values

# Overall metrics
overall_mse = float(np.mean((y_actual - y_pred) ** 2))
overall_smape = smape_mean(y_actual, y_pred)
overall_mae = float(np.mean(np.abs(y_actual - y_pred)))
overall_rmse = float(np.sqrt(overall_mse))

print("\n=== Overall Test Results ===")
print(f"MSE Loss:  {overall_mse:.6f}")
print(f"SMAPE:     {overall_smape:.6f}")
print(f"MAE:       {overall_mae:.6f}")
print(f"RMSE:      {overall_rmse:.6f}")

# Weekly metrics (calendar week)
print("\n=== Weekly Metrics ===")
results_df["Week"] = results_df["Time"].dt.isocalendar().week
weekly_results = []
for week, group in results_df.groupby("Week"):
    week_actual = group["Actual"].values
    week_pred = group["Prediction"].values
    week_rmse = float(np.sqrt(np.mean((week_actual - week_pred) ** 2)))
    week_mae = float(np.mean(np.abs(week_actual - week_pred)))
    week_smape = smape_mean(week_actual, week_pred)
    weekly_results.append({
        "week": int(week),
        "rmse": week_rmse,
        "mae": week_mae,
        "smape": week_smape,
        "n_samples": len(group),
    })
    print(f"Week {week:02d}: RMSE={week_rmse:.4f}, MAE={week_mae:.4f}, SMAPE={week_smape:.4f}")

# Daily metrics
print("\n=== Daily Metrics (by Date) ===")
results_df["Date"] = results_df["Time"].dt.date
daily_results = []
for date, group in results_df.groupby("Date"):
    day_actual = group["Actual"].values
    day_pred = group["Prediction"].values
    day_rmse = float(np.sqrt(np.mean((day_actual - day_pred) ** 2)))
    day_mae = float(np.mean(np.abs(day_actual - day_pred)))
    day_smape = smape_mean(day_actual, day_pred)
    daily_results.append({
        "date": str(date),
        "rmse": day_rmse,
        "mae": day_mae,
        "smape": day_smape,
        "n_samples": len(group),
    })
    print(f"{date}: RMSE={day_rmse:.4f}, MAE={day_mae:.4f}, SMAPE={day_smape:.4f}")

# Day-of-forecast metrics (Day 1..7 inside each predicted week)
print("\n=== Day-of-Forecast Metrics (within each 168-hour period) ===")
day_of_forecast_smapes = {}
for day_num in range(1, 8):
    smapes = [
        day_smapes_per_week[w].get(day_num)
        for w in day_smapes_per_week
        if day_num in day_smapes_per_week[w]
    ]
    smapes = [s for s in smapes if s is not None and not np.isnan(s)]
    day_of_forecast_smapes[day_num] = float(np.mean(smapes)) if smapes else float("nan")
    print(f"Day {day_num} (hours {(day_num-1)*24}-{day_num*24-1}): avg SMAPE={day_of_forecast_smapes[day_num]:.4f}")

# Average weekly/daily metrics
avg_weekly_rmse = float(np.mean([r["rmse"] for r in weekly_results]))
avg_weekly_mae = float(np.mean([r["mae"] for r in weekly_results]))
avg_weekly_smape = float(np.mean([r["smape"] for r in weekly_results]))

avg_daily_rmse = float(np.mean([r["rmse"] for r in daily_results]))
avg_daily_mae = float(np.mean([r["mae"] for r in daily_results]))
avg_daily_smape = float(np.mean([r["smape"] for r in daily_results]))

print("\n=== Average Weekly Metrics ===")
print(f"Avg Weekly RMSE: {avg_weekly_rmse:.6f}")
print(f"Avg Weekly MAE:  {avg_weekly_mae:.6f}")
print(f"Avg Weekly SMAPE: {avg_weekly_smape:.6f}")

print("\n=== Average Daily Metrics ===")
print(f"Avg Daily RMSE: {avg_daily_rmse:.6f}")
print(f"Avg Daily MAE:  {avg_daily_mae:.6f}")
print(f"Avg Daily SMAPE: {avg_daily_smape:.6f}")

# Save all hourly predictions to CSV
output_root = Path(project_root) / "Deep learners" / "LSTM Autoencoder"
output_root.mkdir(parents=True, exist_ok=True)
predictions_csv_path = output_root / f"{WANDB_RUN_NAME}_test_predictions.csv"
results_df.to_csv(predictions_csv_path, index=False, sep=";", decimal=".")
print(f"\nHourly predictions saved to: {predictions_csv_path}")

# Log metrics
test_run.log({
    "test_overall_MSE_loss": overall_mse,
    "test_overall_SMAPE": overall_smape,
    "test_overall_MAE": overall_mae,
    "test_overall_RMSE": overall_rmse,
    "test_avg_weekly_RMSE": avg_weekly_rmse,
    "test_avg_weekly_MAE": avg_weekly_mae,
    "test_avg_weekly_SMAPE": avg_weekly_smape,
    "test_avg_daily_RMSE": avg_daily_rmse,
    "test_avg_daily_MAE": avg_daily_mae,
    "test_avg_daily_SMAPE": avg_daily_smape,
    "test_avg_smape_day_1": day_of_forecast_smapes.get(1, float("nan")),
    "test_avg_smape_day_2": day_of_forecast_smapes.get(2, float("nan")),
    "test_avg_smape_day_3": day_of_forecast_smapes.get(3, float("nan")),
    "test_avg_smape_day_4": day_of_forecast_smapes.get(4, float("nan")),
    "test_avg_smape_day_5": day_of_forecast_smapes.get(5, float("nan")),
    "test_avg_smape_day_6": day_of_forecast_smapes.get(6, float("nan")),
    "test_avg_smape_day_7": day_of_forecast_smapes.get(7, float("nan")),
})

weekly_df = pd.DataFrame(weekly_results)
daily_df = pd.DataFrame(daily_results)
test_run.log({
    "test_weekly_metrics": wandb.Table(dataframe=weekly_df),
    "test_daily_metrics": wandb.Table(dataframe=daily_df),
    "test_predictions_sample": wandb.Table(dataframe=results_df.head(100)),
})

test_run.summary.update({
    "test_overall_MSE_loss": overall_mse,
    "test_overall_SMAPE": overall_smape,
    "test_overall_MAE": overall_mae,
    "test_overall_RMSE": overall_rmse,
    "test_avg_weekly_RMSE": avg_weekly_rmse,
    "test_avg_weekly_MAE": avg_weekly_mae,
    "test_avg_weekly_SMAPE": avg_weekly_smape,
    "test_avg_daily_RMSE": avg_daily_rmse,
    "test_avg_daily_MAE": avg_daily_mae,
    "test_avg_daily_SMAPE": avg_daily_smape,
    "test_samples": int(len(results_df)),
    "test_period_start": str(TEST_START),
    "test_period_end": str(TEST_END),
    "n_weeks": len(weekly_results),
    "n_days": len(daily_results),
})

print(f"\nTest predictions and metrics logged to W&B run: {WANDB_TEST_RUN_NAME}")
wandb.finish()
